# Navigation


In [ ]:

import math, random

from dataclasses import dataclass

from typing import Dict, List, Tuple, Optional

import numpy as np

import pandas as pd

import matplotlib.pyplot as plt

from matplotlib.animation import FuncAnimation

from IPython.display import HTML, display

try:

    from ratinabox import Environment, Agent

    RIA_AVAILABLE = True

except Exception:

    Environment = None

    Agent = None

    RIA_AVAILABLE = False

SEED = 42

rng = np.random.default_rng(SEED)

random.seed(SEED)

@dataclass

class Params:

    env_size_cm: float = 150.0

    speed_cm_s: float = 60.0

    theta_hz: float = 10.0

    phases_per_theta: int = 3

    encounter_radius_cm: float = 10.0

    exploration_turn_deg: float = 30.0

    n_sensory_per_cue: int = 15

    sensory_L_cm: float = 14.0

    n_ec: int = 1000

    n_pc: int = 250

    n_sc: int = 250

    pc_group_size: int = 50

    sc_group_size: int = 25

    connectivity_ec_pc: float = 0.5

    connectivity_pc_sc: float = 0.5

    cin_initial: float = 1.0

    csh_pc: float = 1.0

    csh_sc: float = 0.5

    n_goal_dirs: int = 8

    agent_radius_cm: float = 2.0

P = Params()

DT = 1.0 / (P.theta_hz * P.phases_per_theta)

THETA_DT = 1.0 / P.theta_hz

STEP_CM = P.speed_cm_s * THETA_DT

PHASES = ['early', 'middle', 'late']

GOAL_DIRS = [

    ('E', 0.0),

    ('NE', np.pi / 4),

    ('N', np.pi / 2),

    ('NW', 3 * np.pi / 4),

    ('W', np.pi),

    ('SW', -3 * np.pi / 4),

    ('S', -np.pi / 2),

    ('SE', -np.pi / 4),

]

def wrap_angle(a: float) -> float:

    return (a + np.pi) % (2 * np.pi) - np.pi

def circular_mean_angle(a: float, b: float, wa: float = 1.0, wb: float = 1.0) -> float:

    x = wa * np.cos(a) + wb * np.cos(b)

    y = wa * np.sin(a) + wb * np.sin(b)

    return math.atan2(y, x)

def angle_to_vector(angle: float) -> np.ndarray:

    return np.array([np.cos(angle), np.sin(angle)], dtype=float)

def vector_to_angle(v: np.ndarray) -> float:

    return math.atan2(float(v[1]), float(v[0]))

def l_wall_segments_cm() -> List[np.ndarray]:

    return [

        np.array([[75.0, 30.0], [75.0, 110.0]], dtype=float),

        np.array([[75.0, 110.0], [125.0, 110.0]], dtype=float),

    ]

def make_ratinabox_env_l_wall(P: Params = P):

    if not RIA_AVAILABLE:

        return None

    env = Environment(params={

        'dimensionality': '2D',

        'boundary_conditions': 'solid',

        'scale': P.env_size_cm / 100.0,

        'aspect': 1,

    })

    for wall_cm in l_wall_segments_cm():

        env.add_wall(wall_cm / 100.0)

    return env

def _goal_to_xy(goal):

    if goal is None:

        return None

    if hasattr(goal, "position"):

        return np.asarray(goal.position, dtype=float)

    return np.asarray(goal, dtype=float)

def plot_world(cues=None, goal=None, starts=None, ax=None, title='L-wall arena'):

    if ax is None:

        fig, ax = plt.subplots(figsize=(6,6))

    goal_xy = _goal_to_xy(goal)

    ax.set_xlim(-5, P.env_size_cm + 5)

    ax.set_ylim(-5, P.env_size_cm + 5)

    ax.set_aspect('equal')

    ax.plot([0, P.env_size_cm, P.env_size_cm, 0, 0], [0, 0, P.env_size_cm, P.env_size_cm, 0], 'k-', lw=2)

    for wall in l_wall_segments_cm():

        ax.plot(wall[:,0], wall[:,1], 'k-', lw=5, solid_capstyle='round')

    if cues is not None:

        cues = np.asarray(cues, dtype=float)

        ax.scatter(cues[:,0], cues[:,1], marker='+', s=90, label='cues')

    if goal_xy is not None:

        ax.scatter([goal_xy[0]], [goal_xy[1]], marker='x', s=140, label='goal')

    if starts is not None:

        starts = np.array(starts)

        ax.scatter(starts[:,0], starts[:,1], marker='o', s=45, label='starts')

    ax.set_xlabel('x, cm')

    ax.set_ylabel('y, cm')

    ax.set_title(title)

    if any(x is not None for x in [cues, goal_xy, starts]):

        ax.legend(loc='best')

    return ax

def orient(a,b,c):

    return (b[0]-a[0])*(c[1]-a[1]) - (b[1]-a[1])*(c[0]-a[0])

def on_segment(a,b,c, eps=1e-9):

    return min(a[0], b[0])-eps <= c[0] <= max(a[0], b[0])+eps and min(a[1], b[1])-eps <= c[1] <= max(a[1], b[1])+eps

def segments_intersect(a,b,c,d, eps=1e-9):

    o1 = orient(a,b,c); o2 = orient(a,b,d); o3 = orient(c,d,a); o4 = orient(c,d,b)

    if abs(o1) < eps and on_segment(a,b,c): return True

    if abs(o2) < eps and on_segment(a,b,d): return True

    if abs(o3) < eps and on_segment(c,d,a): return True

    if abs(o4) < eps and on_segment(c,d,b): return True

    return (o1*o2 < 0) and (o3*o4 < 0)

def point_segment_distance(p, a, b):

    ab = b - a

    denom = np.dot(ab, ab)

    if denom < 1e-12:

        return float(np.linalg.norm(p-a))

    t = np.clip(np.dot(p-a, ab) / denom, 0, 1)

    proj = a + t*ab

    return float(np.linalg.norm(p-proj))

def is_inside_bounds(pos, P=P):

    r = P.agent_radius_cm

    return r <= pos[0] <= P.env_size_cm-r and r <= pos[1] <= P.env_size_cm-r

def step_is_valid(pos, new_pos, walls=None, P=P, margin=True):

    if walls is None:

        walls = l_wall_segments_cm()

    if not is_inside_bounds(new_pos, P):

        return False

    for wall in walls:

        a, b = wall

        if segments_intersect(pos, new_pos, a, b):

            return False

        if margin and point_segment_distance(new_pos, a, b) < P.agent_radius_cm:

            return False

    return True

def choose_wall_aware_heading(pos, desired_heading, step_cm=STEP_CM, walls=None, P=P):

    offsets = [0, 10, -10, 20, -20, 30, -30, 45, -45, 60, -60, 90, -90, 120, -120, 150, -150, 180]

    best = None

    for off in offsets:

        h = wrap_angle(desired_heading + np.deg2rad(off))

        new_pos = pos + step_cm * angle_to_vector(h)

        if step_is_valid(pos, new_pos, walls, P):

            return h, False

    return desired_heading, True

def make_perimeter_cues(n_per_side: int = 4, size: float = P.env_size_cm) -> np.ndarray:

    cues = []

    xs = np.linspace(0, size, n_per_side, endpoint=False) + size / (2*n_per_side)

    ys = np.linspace(0, size, n_per_side, endpoint=False) + size / (2*n_per_side)

    for x in xs: cues.append([x, size])

    for y in ys: cues.append([size, y])

    for x in xs[::-1]: cues.append([x, 0])

    for y in ys[::-1]: cues.append([0, y])

    return np.array(cues, dtype=float)

def make_wall_cues() -> np.ndarray:

    return np.array([

        [75.0, 30.0], [75.0, 110.0], [125.0, 110.0],

        [75.0, 70.0], [100.0, 110.0]

    ], dtype=float)

def make_cues(mode='perimeter') -> np.ndarray:

    base = make_perimeter_cues()

    if mode == 'perimeter':

        return base

    if mode == 'perimeter_plus_wall':

        return np.vstack([base, make_wall_cues()])

    raise ValueError(mode)

@dataclass

class ECPair:

    sensory_a: int

    sensory_b: int

    cue_a: int

    cue_b: int

@dataclass

class GoalMemory:

    name: str

    position: np.ndarray

    W_sc_goal: np.ndarray

    train_norm: np.ndarray

class BurgessOkeefeNetwork:

    def __init__(self, cues: np.ndarray, P: Params = P, seed: int = SEED):

        self.cues = np.array(cues, dtype=float)

        self.P = P

        self.rng = np.random.default_rng(seed)

        self.ec_pairs = self.build_ec_pairs()

        self.mask_ec_pc, self.W_ec_pc, self.M_ec_pc = self.make_connectivity_and_weights(P.n_ec, P.n_pc, P.connectivity_ec_pc, P.cin_initial)

        self.mask_pc_sc, self.W_pc_sc, self.M_pc_sc = self.make_connectivity_and_weights(P.n_pc, P.n_sc, P.connectivity_pc_sc, P.cin_initial)

        self.goal_vectors = np.array([angle_to_vector(a) for _, a in GOAL_DIRS])

    def sensory_activation(self, pos: np.ndarray) -> np.ndarray:

        L = self.P.sensory_L_cm

        n_per = self.P.n_sensory_per_cue

        dists = np.linalg.norm(self.cues - pos[None,:], axis=1)

        activations = []

        for d in dists:

            for i in range(n_per):

                preferred = i * L

                delta = abs(d - preferred)

                if delta > 6 * L:

                    n = 0

                else:

                    n = int(np.floor((7 * L - delta) / (2 * L)))

                    n = max(0, min(3, n))

                activations.append(n)

        return np.array(activations, dtype=np.float32)

    def build_ec_pairs(self) -> List[ECPair]:

        n_cues = len(self.cues)

        n_per = self.P.n_sensory_per_cue

        L = self.P.sensory_L_cm

        pairs = []

        for _ in range(self.P.n_ec):

            a, b = self.rng.choice(n_cues, size=2, replace=False)

            s = np.linalg.norm(self.cues[a] - self.cues[b])

            centre_i = s / (2*L)

            ia = int(np.clip(round(centre_i + self.rng.normal(0,2.0)), 0, n_per-1))

            ib = int(np.clip(round(centre_i + self.rng.normal(0,2.0)), 0, n_per-1))

            pairs.append(ECPair(a*n_per + ia, b*n_per + ib, int(a), int(b)))

        self.pair_sensory_a = np.array([p.sensory_a for p in pairs], dtype=np.int32)

        self.pair_sensory_b = np.array([p.sensory_b for p in pairs], dtype=np.int32)

        self.pair_cue_a = np.array([p.cue_a for p in pairs], dtype=np.int32)

        self.pair_cue_b = np.array([p.cue_b for p in pairs], dtype=np.int32)

        self.pair_centroids = 0.5 * (self.cues[self.pair_cue_a] + self.cues[self.pair_cue_b])

        return pairs

    def ec_activation_for_phase(self, sensory, pos, heading, phase):

        d = self.pair_centroids - pos[None, :]

        angles = np.arctan2(d[:,1], d[:,0])

        alpha = np.abs((angles - heading + np.pi) % (2*np.pi) - np.pi)

        if phase == 'late':

            phase_mask = alpha < np.deg2rad(60)

        elif phase == 'middle':

            phase_mask = (alpha >= np.deg2rad(60)) & (alpha < np.deg2rad(120))

        elif phase == 'early':

            phase_mask = alpha >= np.deg2rad(120)

        else:

            raise ValueError(phase)

        ni = sensory[self.pair_sensory_a]

        nj = sensory[self.pair_sensory_b]

        ec = np.floor((ni * nj) / 2.0).astype(np.float32)

        ec[~phase_mask] = 0.0

        return ec

    def make_connectivity_and_weights(self, n_pre, n_post, connectivity, cin_initial):

        mask = self.rng.random((n_post, n_pre)) < connectivity

        p_on = cin_initial / max(1, int(connectivity*n_pre))

        W = (mask & (self.rng.random((n_post,n_pre)) < p_on)).astype(np.float32)

        M = np.zeros(n_post, dtype=np.float32)

        return mask, W, M

    @staticmethod

    def competitive_layer_activation(pre, mask, W, M, group_size, csh):

        input_sum = W @ pre

        h = input_sum / (csh * (1.0 + M))

        out = np.zeros_like(h, dtype=np.float32)

        n = len(h)

        for start in range(0, n, group_size):

            end = min(start + group_size, n)

            group_h = h[start:end]

            if len(group_h) == 0: continue

            top_local = np.argsort(group_h)[::-1][:4]

            for rank, local_idx in enumerate(top_local, start=1):

                max_spikes = 5 - rank

                val = min(group_h[local_idx], max_spikes)

                out[start + local_idx] = np.floor(val)

        return out

    @staticmethod

    def hebbian_binary_update(W, mask, M, pre, post):

        pre_max = pre >= 4

        post_max = post >= 4

        if not pre_max.any() or not post_max.any(): return 0

        candidate = np.zeros_like(W, dtype=bool)

        candidate[np.ix_(post_max, pre_max)] = True

        new_on = candidate & mask & (W == 0)

        if not new_on.any(): return 0

        W[new_on] = 1.0

        per_post = new_on.sum(axis=1).astype(np.float32)

        M += per_post

        return int(per_post.sum())

    def network_step(self, pos, heading, phase, learn=True):

        sensory = self.sensory_activation(pos)

        ec = self.ec_activation_for_phase(sensory, pos, heading, phase)

        pc = self.competitive_layer_activation(ec, self.mask_ec_pc, self.W_ec_pc, self.M_ec_pc, self.P.pc_group_size, self.P.csh_pc)

        sc = self.competitive_layer_activation(pc, self.mask_pc_sc, self.W_pc_sc, self.M_pc_sc, self.P.sc_group_size, self.P.csh_sc)

        updates_ec_pc = updates_pc_sc = 0

        if learn:

            updates_ec_pc = self.hebbian_binary_update(self.W_ec_pc, self.mask_ec_pc, self.M_ec_pc, ec, pc)

            updates_pc_sc = self.hebbian_binary_update(self.W_pc_sc, self.mask_pc_sc, self.M_pc_sc, pc, sc)

        return {'sensory':sensory, 'ec':ec, 'pc':pc, 'sc':sc, 'updates_ec_pc':updates_ec_pc, 'updates_pc_sc':updates_pc_sc}

    def explore_environment(self, seconds=30.0, start_pos=None, start_heading=None, walls=None):

        if walls is None: walls = l_wall_segments_cm()

        pos = np.array(start_pos if start_pos is not None else [self.P.env_size_cm/2, self.P.env_size_cm/2], dtype=float)

        heading = float(start_heading if start_heading is not None else self.rng.uniform(-np.pi, np.pi))

        n_cycles = int(seconds * self.P.theta_hz)

        rows = []

        for cycle in range(n_cycles):

            total_updates_ec_pc = total_updates_pc_sc = 0

            for phase in PHASES:

                out = self.network_step(pos, heading, phase, learn=True)

                total_updates_ec_pc += out['updates_ec_pc']

                total_updates_pc_sc += out['updates_pc_sc']

            rows.append({'cycle':cycle, 't_s':cycle/self.P.theta_hz, 'x':pos[0], 'y':pos[1], 'heading':heading,

                         'updates_ec_pc':total_updates_ec_pc, 'updates_pc_sc':total_updates_pc_sc})

            for _attempt in range(20):

                turn = self.rng.uniform(-np.deg2rad(self.P.exploration_turn_deg), np.deg2rad(self.P.exploration_turn_deg))

                h_try = wrap_angle(heading + turn)

                new_pos = pos + STEP_CM * angle_to_vector(h_try)

                if step_is_valid(pos, new_pos, walls, self.P):

                    heading = h_try

                    pos = new_pos

                    break

            else:

                heading = wrap_angle(heading + np.pi/2)

        return pd.DataFrame(rows)

    def goal_activation_from_sc(self, sc, memory):

        W = memory.W_sc_goal

        input_sum = W @ sc

        m = W.sum(axis=1)

        raw = 25.0 * input_sum / (1.0 + m)

        return np.floor(raw).astype(np.float32)

    def learn_goal(self, name, goal_pos):

        W_sc_goal = np.zeros((self.P.n_goal_dirs, self.P.n_sc), dtype=np.float32)

        memory = GoalMemory(name=name, position=np.array(goal_pos, dtype=float), W_sc_goal=W_sc_goal, train_norm=np.ones(self.P.n_goal_dirs))

        for d_idx, (label, direction_angle) in enumerate(GOAL_DIRS):

            for phase in PHASES:

                out = self.network_step(memory.position, direction_angle, phase, learn=False)

                if phase == 'late':

                    active_sc = out['sc'] > 0

                    W_sc_goal[d_idx, active_sc] = 1.0

            out_late = self.network_step(memory.position, direction_angle, 'late', learn=False)

            act = self.goal_activation_from_sc(out_late['sc'], memory)[d_idx]

            memory.train_norm[d_idx] = max(1.0, act)

        return memory

    def goal_activations_for_cycle(self, pos, heading, memory):

        total = np.zeros(self.P.n_goal_dirs, dtype=np.float32)

        for phase in PHASES:

            out = self.network_step(pos, heading, phase, learn=False)

            total += self.goal_activation_from_sc(out['sc'], memory)

        return total

    def population_vector_from_goal_cells(self, goal_act, memory):

        weights = goal_act / memory.train_norm

        return weights @ self.goal_vectors

    def direction_to_goal(self, pos, heading, memory):

        goal_act = self.goal_activations_for_cycle(pos, heading, memory)

        v_from_goal_to_rat = self.population_vector_from_goal_cells(goal_act, memory)

        if np.linalg.norm(v_from_goal_to_rat) < 1e-6:

            desired = heading

        else:

            desired = vector_to_angle(-v_from_goal_to_rat)

        return desired, {'goal_act':goal_act, 'v_from_goal_to_rat':v_from_goal_to_rat}

    def navigate_to_goal(self, start_pos, memory, start_heading=0.0, max_seconds=15.0, momentum=1.0, walls=None, controller='baseline'):

        if walls is None: walls = l_wall_segments_cm()

        pos = np.array(start_pos, dtype=float)

        heading = float(start_heading)

        n_cycles = int(max_seconds * self.P.theta_hz)

        rows = []

        blocked_count = 0

        for cycle in range(n_cycles):

            desired, info = self.direction_to_goal(pos, heading, memory)

            smoothed = circular_mean_angle(heading, desired, wa=momentum, wb=1.0)

            if controller == 'wall_aware':

                chosen_heading, stuck = choose_wall_aware_heading(pos, smoothed, STEP_CM, walls, self.P)

            elif controller == 'baseline':

                chosen_heading, stuck = smoothed, False

            else:

                raise ValueError(controller)

            new_pos = pos + STEP_CM * angle_to_vector(chosen_heading)

            blocked = not step_is_valid(pos, new_pos, walls, self.P)

            if blocked:

                blocked_count += 1

                if controller == 'baseline':

                    chosen_heading = wrap_angle(heading + self.rng.choice([-1,1]) * np.pi/2)

                    new_pos = pos.copy()

                else:

                    new_pos = pos.copy()

            heading = chosen_heading

            dist_to_goal = np.linalg.norm(pos - memory.position)

            rows.append({'cycle':cycle, 't_s':cycle/self.P.theta_hz, 'x':pos[0], 'y':pos[1], 'heading':heading,

                         'desired':desired, 'chosen_heading':chosen_heading, 'dist_to_goal':dist_to_goal,

                         'goal_vector_norm':np.linalg.norm(info['v_from_goal_to_rat']), 'blocked':blocked,

                         'blocked_count':blocked_count,

                         **{f'GC_{label}': info['goal_act'][i] for i,(label,_) in enumerate(GOAL_DIRS)}})

            if dist_to_goal <= self.P.encounter_radius_cm:

                break

            pos = new_pos

        return pd.DataFrame(rows)

def eight_edge_starts(size=P.env_size_cm, margin=10.0):

    return [np.array([margin, margin]), np.array([size/2, margin]), np.array([size-margin, margin]),

            np.array([size-margin, size/2]), np.array([size-margin, size-margin]), np.array([size/2, size-margin]),

            np.array([margin, size-margin]), np.array([margin, size/2])]

def path_length_cm(df):

    pts = df[['x','y']].values

    if len(pts) < 2: return 0.0

    return float(np.linalg.norm(np.diff(pts, axis=0), axis=1).sum())

def run_many_starts(net, memory, starts, controller='baseline', walls=None, max_seconds=15):

    rows = []

    navs = []

    for i, st in enumerate(starts):

        df = net.navigate_to_goal(st, memory, start_heading=net.rng.uniform(-np.pi,np.pi), max_seconds=max_seconds, walls=walls, controller=controller)

        df['run'] = i

        navs.append(df)

        rows.append({'run':i, 'controller':controller, 'success': bool(df.dist_to_goal.iloc[-1] <= P.encounter_radius_cm),

                     'time_s': float(df.t_s.iloc[-1]), 'final_dist_cm': float(df.dist_to_goal.iloc[-1]),

                     'path_length_cm': path_length_cm(df), 'blocked_steps': int(df.blocked.sum())})

    return navs, pd.DataFrame(rows)

def plot_navs(navs, goal, cues, title):

    fig, ax = plt.subplots(figsize=(6,6))

    plot_world(cues=cues, goal=goal, ax=ax, title=title)

    for df in navs:

        ax.plot(df.x, df.y, lw=1.6)

    return fig, ax

def field_map_for_cell(net, layer, cell_idx, heading=0.0, phase='middle', grid_n=35, memory=None):

    xs = np.linspace(0, P.env_size_cm, grid_n)

    ys = np.linspace(0, P.env_size_cm, grid_n)

    fmap = np.full((grid_n, grid_n), np.nan, dtype=float)

    walls = l_wall_segments_cm()

    for iy, y in enumerate(ys):

        for ix, x in enumerate(xs):

            pos = np.array([x,y], dtype=float)

            if not is_inside_bounds(pos, P):

                continue

            if any(point_segment_distance(pos, w[0], w[1]) < P.agent_radius_cm for w in walls):

                continue

            out = net.network_step(pos, heading, phase, learn=False)

            if layer == 'ec': fmap[iy,ix] = out['ec'][cell_idx]

            elif layer == 'pc': fmap[iy,ix] = out['pc'][cell_idx]

            elif layer == 'sc': fmap[iy,ix] = out['sc'][cell_idx]

            elif layer == 'goal': fmap[iy,ix] = net.goal_activations_for_cycle(pos, heading, memory)[cell_idx]

            else: raise ValueError(layer)

    return xs, ys, fmap

def find_active_cell(net, layer='pc', samples=80, phase='middle'):

    n = {'ec':P.n_ec, 'pc':P.n_pc, 'sc':P.n_sc}[layer]

    max_seen = np.zeros(n)

    for _ in range(samples):

        for _try in range(50):

            pos = net.rng.uniform(P.agent_radius_cm, P.env_size_cm-P.agent_radius_cm, size=2)

            if all(point_segment_distance(pos, w[0], w[1]) > P.agent_radius_cm for w in l_wall_segments_cm()):

                break

        out = net.network_step(pos, 0.0, phase, learn=False)

        max_seen = np.maximum(max_seen, out[layer])

    return int(np.argmax(max_seen))

def plot_field_map(xs, ys, fmap, title, cues=None, goal=None):

    fig, ax = plt.subplots(figsize=(5.5,4.8))

    im = ax.imshow(fmap, origin='lower', extent=[0,P.env_size_cm,0,P.env_size_cm], aspect='equal')

    for wall in l_wall_segments_cm():

        ax.plot(wall[:,0], wall[:,1], 'w-', lw=4)

        ax.plot(wall[:,0], wall[:,1], 'k-', lw=2)

    if cues is not None: ax.scatter(cues[:,0], cues[:,1], marker='+', s=45)

    goal_xy = _goal_to_xy(goal)

    if goal_xy is not None: ax.scatter([goal_xy[0]], [goal_xy[1]], marker='x', s=100)

    ax.set_title(title); ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm')

    fig.colorbar(im, ax=ax, label='activation')

    return fig, ax

def animate_navigation(nav_df, goal, cues):

    fig, ax = plt.subplots(figsize=(6,6))

    plot_world(cues=cues, goal=goal, ax=ax, title='Navigation animation')

    path_line, = ax.plot([], [], lw=2)

    dot, = ax.plot([], [], 'o', ms=8)

    text = ax.text(0.02, 0.98, '', transform=ax.transAxes, va='top')

    xs = nav_df.x.values; ys = nav_df.y.values; times = nav_df.t_s.values

    def init():

        path_line.set_data([], []); dot.set_data([], []); text.set_text('')

        return path_line, dot, text

    def update(frame):

        path_line.set_data(xs[:frame+1], ys[:frame+1])

        dot.set_data([xs[frame]], [ys[frame]])

        text.set_text(f't={times[frame]:.1f}s\ndist={nav_df.dist_to_goal.iloc[frame]:.1f}cm')

        return path_line, dot, text

    anim = FuncAnimation(fig, update, frames=len(nav_df), init_func=init, interval=120, blit=False)

    plt.close(fig)

    return HTML(anim.to_jshtml())

def save_notebook_outputs(output_dir, explore_df=None, summary=None, comparison=None,

                          navs_baseline=None, navs_wall=None, navs_wc_wall=None,

                          cues=None, cues_wall=None, goal=None):

    from pathlib import Path

    import zipfile

    out = Path(output_dir)

    out.mkdir(parents=True, exist_ok=True)

    goal_xy = _goal_to_xy(goal)

    if explore_df is not None:

        explore_df.to_csv(out / 'explore_trajectory.csv', index=False)

    if summary is not None:

        summary.to_csv(out / 'summary_baseline_vs_wallaware.csv', index=False)

    if comparison is not None:

        comparison.to_csv(out / 'comparison_all_models.csv', index=False)

    def save_nav_list(navs, prefix):

        if navs is None:

            return

        d = out / prefix

        d.mkdir(exist_ok=True)

        for i, df in enumerate(navs):

            df.to_csv(d / f'run_{i:02d}.csv', index=False)

    save_nav_list(navs_baseline, 'navs_baseline')

    save_nav_list(navs_wall, 'navs_wallaware')

    save_nav_list(navs_wc_wall, 'navs_wallcues_wallaware')

    if navs_baseline is not None:

        fig, ax = plot_navs(navs_baseline, goal_xy, cues, 'Baseline: goal-vector without wall-aware filter')

        fig.savefig(out / 'trajectories_baseline.png', dpi=180, bbox_inches='tight')

        plt.close(fig)

    if navs_wall is not None:

        fig, ax = plot_navs(navs_wall, goal_xy, cues, 'Wall-aware controller')

        fig.savefig(out / 'trajectories_wallaware.png', dpi=180, bbox_inches='tight')

        plt.close(fig)

    if navs_wc_wall is not None:

        fig, ax = plot_navs(navs_wc_wall, goal_xy, cues_wall if cues_wall is not None else cues, 'Wall cues + wall-aware controller')

        fig.savefig(out / 'trajectories_wallcues_wallaware.png', dpi=180, bbox_inches='tight')

        plt.close(fig)

    if navs_wall is not None and len(navs_wall) > 0:

        html = animate_navigation(navs_wall[0], goal_xy, cues)

        (out / 'animation_wallaware_run0.html').write_text(html.data, encoding='utf-8')

    zip_path = out.with_suffix('.zip')

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:

        for fp in out.rglob('*'):

            zf.write(fp, fp.relative_to(out.parent))

    return str(zip_path)

from pathlib import Path

import zipfile

def make_lwall_waypoints(P: Params = P, clearance_cm: float = 16.0) -> Dict[str, np.ndarray]:

    raw = {

        'v_low_left':  np.array([75.0 - clearance_cm, 30.0 - clearance_cm], dtype=float),

        'v_low_right': np.array([75.0 + clearance_cm, 30.0 - clearance_cm], dtype=float),

        'v_mid_left':  np.array([75.0 - clearance_cm, 70.0], dtype=float),

        'v_mid_right': np.array([75.0 + clearance_cm, 70.0], dtype=float),

        'corner_upper_left':  np.array([75.0 - clearance_cm, 110.0 + clearance_cm], dtype=float),

        'corner_lower_left':  np.array([75.0 - clearance_cm, 110.0 - clearance_cm], dtype=float),

        'corner_upper_right': np.array([75.0 + clearance_cm, 110.0 + clearance_cm], dtype=float),

        'corner_lower_right': np.array([75.0 + clearance_cm, 110.0 - clearance_cm], dtype=float),

        'h_right_below': np.array([125.0 + clearance_cm, 110.0 - clearance_cm], dtype=float),

        'h_right_above': np.array([125.0 + clearance_cm, 110.0 + clearance_cm], dtype=float),

        'h_mid_below':   np.array([100.0, 110.0 - clearance_cm], dtype=float),

        'h_mid_above':   np.array([100.0, 110.0 + clearance_cm], dtype=float),

    }

    return {k: v for k, v in raw.items() if point_is_valid(v, margin=True)}

def point_is_valid(pos: np.ndarray, walls=None, P: Params = P, margin: bool = True) -> bool:

    if walls is None:

        walls = l_wall_segments_cm()

    pos = np.asarray(pos, dtype=float)

    if not is_inside_bounds(pos, P):

        return False

    if margin:

        for w in walls:

            if point_segment_distance(pos, w[0], w[1]) < P.agent_radius_cm:

                return False

    return True

def path_is_clear(a: np.ndarray, b: np.ndarray, walls=None, P: Params = P,

                  margin: bool = True, sample_step_cm: float = 2.0) -> bool:

    if walls is None:

        walls = l_wall_segments_cm()

    a = np.asarray(a, dtype=float)

    b = np.asarray(b, dtype=float)

    if not point_is_valid(a, walls, P, margin) or not point_is_valid(b, walls, P, margin):

        return False

    for wall in walls:

        if segments_intersect(a, b, wall[0], wall[1]):

            return False

    if margin:

        dist = float(np.linalg.norm(b - a))

        n = max(2, int(np.ceil(dist / sample_step_cm)))

        for t in np.linspace(0.0, 1.0, n):

            p = (1.0 - t) * a + t * b

            if not is_inside_bounds(p, P):

                return False

            for wall in walls:

                if point_segment_distance(p, wall[0], wall[1]) < P.agent_radius_cm:

                    return False

    return True

def _dijkstra(nodes: List[np.ndarray], edges: Dict[int, List[Tuple[int, float]]]):

    import heapq

    n = len(nodes)

    dist = [float('inf')] * n

    prev = [None] * n

    dist[0] = 0.0

    heap = [(0.0, 0)]

    while heap:

        d, u = heapq.heappop(heap)

        if d != dist[u]:

            continue

        if u == n - 1:

            break

        for v, w in edges.get(u, []):

            nd = d + w

            if nd < dist[v]:

                dist[v] = nd

                prev[v] = u

                heapq.heappush(heap, (nd, v))

    if not np.isfinite(dist[-1]):

        return None, float('inf')

    path = []

    u = n - 1

    while u is not None:

        path.append(u)

        u = prev[u]

    return path[::-1], dist[-1]

def visibility_graph_path(pos: np.ndarray, goal_pos: np.ndarray, waypoint_dict: Dict[str, np.ndarray],

                          walls=None, P: Params = P) -> Tuple[List[str], List[np.ndarray], float]:

    if walls is None:

        walls = l_wall_segments_cm()

    valid_waypoints = [(name, np.asarray(w, dtype=float))

                       for name, w in waypoint_dict.items()

                       if point_is_valid(w, walls, P, margin=True)]

    labels = ['current'] + [x[0] for x in valid_waypoints] + ['goal']

    nodes = [np.asarray(pos, dtype=float)] + [x[1] for x in valid_waypoints] + [np.asarray(goal_pos, dtype=float)]

    edges = {i: [] for i in range(len(nodes))}

    for i in range(len(nodes)):

        for j in range(i + 1, len(nodes)):

            if path_is_clear(nodes[i], nodes[j], walls, P, margin=True):

                w = float(np.linalg.norm(nodes[j] - nodes[i]))

                edges[i].append((j, w))

                edges[j].append((i, w))

    path_idx, total = _dijkstra(nodes, edges)

    if path_idx is None:

        return [], [], float('inf')

    return [labels[i] for i in path_idx], [nodes[i] for i in path_idx], total

def select_next_target(pos: np.ndarray, goal_pos: np.ndarray, waypoint_dict: Dict[str, np.ndarray],

                       walls=None, P: Params = P) -> Tuple[str, np.ndarray, List[str], float]:

    if walls is None:

        walls = l_wall_segments_cm()

    goal_pos = np.asarray(goal_pos, dtype=float)

    if path_is_clear(pos, goal_pos, walls, P, margin=True):

        return 'goal', goal_pos, ['current', 'goal'], float(np.linalg.norm(goal_pos - pos))

    labels, coords, total = visibility_graph_path(pos, goal_pos, waypoint_dict, walls, P)

    if len(coords) >= 2:

        return labels[1], coords[1], labels, total

    candidates = []

    for name, w in waypoint_dict.items():

        w = np.asarray(w, dtype=float)

        if point_is_valid(w, walls, P, margin=True) and path_is_clear(pos, w, walls, P, margin=True):

            candidates.append((float(np.linalg.norm(pos - w) + np.linalg.norm(w - goal_pos)), name, w))

    if candidates:

        _, name, w = min(candidates, key=lambda x: x[0])

        return name, w, ['current', name, 'goal?'], float(np.linalg.norm(pos - w) + np.linalg.norm(w - goal_pos))

    return 'stuck', np.asarray(pos, dtype=float), ['current'], float('inf')

@dataclass

class TargetManagerConfig:

    waypoint_radius_cm: float = 14.0

    goal_radius_cm: float = P.encounter_radius_cm

    min_lock_steps: int = 8

    stall_patience_steps: int = 18

    min_progress_cm: float = 0.7

    weak_vector_threshold: float = 0.18

class TargetManager:

    def __init__(self, final_goal_pos: np.ndarray, waypoint_dict: Dict[str, np.ndarray],

                 config: TargetManagerConfig = TargetManagerConfig(), walls=None, P: Params = P):

        self.goal_pos = np.asarray(final_goal_pos, dtype=float)

        self.waypoints = waypoint_dict

        self.cfg = config

        self.walls = l_wall_segments_cm() if walls is None else walls

        self.P = P

        self.current_name = None

        self.current_pos = None

        self.path_labels = []

        self.path_estimate = float('inf')

        self.lock_steps_left = 0

        self.best_dist_to_target = float('inf')

        self.stall_count = 0

        self.switch_count = 0

        self.forced_replans = 0

    def _radius_for(self, name: str) -> float:

        return self.cfg.goal_radius_cm if name == 'goal' else self.cfg.waypoint_radius_cm

    def _set_target(self, name, pos, labels, estimate):

        pos = np.asarray(pos, dtype=float)

        if self.current_name is not None and name != self.current_name:

            self.switch_count += 1

        self.current_name = name

        self.current_pos = pos

        self.path_labels = labels

        self.path_estimate = estimate

        self.lock_steps_left = self.cfg.min_lock_steps

        self.best_dist_to_target = float('inf')

        self.stall_count = 0

    def replan(self, pos):

        name, tpos, labels, est = select_next_target(pos, self.goal_pos, self.waypoints, self.walls, self.P)

        self._set_target(name, tpos, labels, est)

        return self.current_name, self.current_pos

    def update(self, pos):

        pos = np.asarray(pos, dtype=float)

        if self.current_name is None:

            self.replan(pos)

        dist = float(np.linalg.norm(pos - self.current_pos))

        reached_current = dist <= self._radius_for(self.current_name)

        if reached_current:

            self.replan(pos)

            dist = float(np.linalg.norm(pos - self.current_pos))

        if self.lock_steps_left <= 0 and self.current_name != 'goal':

            if path_is_clear(pos, self.goal_pos, self.walls, self.P, margin=True):

                self._set_target('goal', self.goal_pos, ['current', 'goal'], float(np.linalg.norm(pos - self.goal_pos)))

                dist = float(np.linalg.norm(pos - self.current_pos))

        if dist < self.best_dist_to_target - self.cfg.min_progress_cm:

            self.best_dist_to_target = dist

            self.stall_count = 0

        else:

            self.stall_count += 1

        if self.lock_steps_left > 0:

            self.lock_steps_left -= 1

        if self.lock_steps_left <= 0 and self.stall_count >= self.cfg.stall_patience_steps:

            self.forced_replans += 1

            self.replan(pos)

        return {

            'target_name': self.current_name,

            'target_pos': self.current_pos.copy(),

            'dist_to_target': float(np.linalg.norm(pos - self.current_pos)),

            'path_labels': ' -> '.join(self.path_labels),

            'path_estimate_cm': self.path_estimate,

            'target_switches': self.switch_count,

            'stall_count': self.stall_count,

            'forced_replans': self.forced_replans,

            'lock_steps_left': self.lock_steps_left,

        }

def choose_progress_aware_heading(pos: np.ndarray, desired_heading: float, target_pos: np.ndarray,

                                  step_cm: float = STEP_CM, walls=None, P: Params = P,

                                  max_offset_deg: int = 180):

    if walls is None:

        walls = l_wall_segments_cm()

    offsets = [0]

    for d in [5,10,15,20,30,40,50,60,75,90,120,150,180]:

        offsets.extend([d, -d])

    best = None

    target_pos = np.asarray(target_pos, dtype=float)

    for off in offsets:

        if abs(off) > max_offset_deg:

            continue

        h = wrap_angle(desired_heading + np.deg2rad(off))

        new_pos = pos + step_cm * angle_to_vector(h)

        if not step_is_valid(pos, new_pos, walls, P):

            continue

        progress_cost = float(np.linalg.norm(new_pos - target_pos))

        angular_cost = 0.04 * abs(off)

        wall_clearance = min(point_segment_distance(new_pos, w[0], w[1]) for w in walls)

        clearance_penalty = 1.5 / max(1.0, wall_clearance)

        score = progress_cost + angular_cost + clearance_penalty

        if best is None or score < best[0]:

            best = (score, h, new_pos, off)

    if best is None:

        return desired_heading, pos.copy(), True, None

    return best[1], best[2], False, best[3]

def learn_waypoint_memories(net: BurgessOkeefeNetwork, waypoint_dict: Dict[str, np.ndarray],

                            walls=None, P: Params = P) -> Dict[str, GoalMemory]:

    if walls is None:

        walls = l_wall_segments_cm()

    memories = {}

    for name, pos in waypoint_dict.items():

        if point_is_valid(pos, walls, P, margin=True):

            memories[name] = net.learn_goal('waypoint_' + name, pos)

    return memories

def memory_direction_to_named_target(net: BurgessOkeefeNetwork, pos: np.ndarray, heading: float,

                                     target_name: str, final_memory: GoalMemory,

                                     waypoint_memories: Optional[Dict[str, GoalMemory]]):

    if target_name == 'goal':

        memory = final_memory

    else:

        if waypoint_memories is None or target_name not in waypoint_memories:

            return None, {'goal_act': np.zeros(net.P.n_goal_dirs), 'v_from_goal_to_rat': np.zeros(2)}, None

        memory = waypoint_memories[target_name]

    desired, info = net.direction_to_goal(pos, heading, memory)

    return desired, info, memory

def navigate_with_target_manager(net: BurgessOkeefeNetwork, start_pos: np.ndarray, final_memory: GoalMemory,

                                 waypoint_dict: Dict[str, np.ndarray], waypoint_memories: Optional[Dict[str, GoalMemory]] = None,

                                 start_heading: float = 0.0, max_seconds: float = 20.0, momentum: float = 0.55,

                                 walls=None, controller: str = 'waypoint_geometric_oracle',

                                 cfg: TargetManagerConfig = TargetManagerConfig()) -> pd.DataFrame:

    if walls is None:

        walls = l_wall_segments_cm()

    if controller not in ['baseline_neural', 'wall_aware_neural', 'waypoint_geometric_oracle', 'waypoint_memory_pure', 'waypoint_memory_hybrid']:

        raise ValueError(controller)

    pos = np.array(start_pos, dtype=float)

    heading = float(start_heading)

    n_cycles = int(max_seconds * net.P.theta_hz)

    rows = []

    blocked_count = 0

    manager = None

    if controller in ['waypoint_geometric_oracle', 'waypoint_memory_pure', 'waypoint_memory_hybrid']:

        manager = TargetManager(final_memory.position, waypoint_dict, config=cfg, walls=walls, P=net.P)

    for cycle in range(n_cycles):

        dist_to_goal = float(np.linalg.norm(pos - final_memory.position))

        if manager is None:

            target_name = 'goal'

            target_pos = final_memory.position.copy()

            target_state = {

                'target_name': 'goal',

                'target_pos': target_pos,

                'dist_to_target': dist_to_goal,

                'path_labels': 'current -> goal_direct_command',

                'path_estimate_cm': float(np.linalg.norm(pos - final_memory.position)),

                'target_switches': 0,

                'stall_count': 0,

                'forced_replans': 0,

                'lock_steps_left': 0,

            }

        else:

            target_state = manager.update(pos)

            target_name = target_state['target_name']

            target_pos = target_state['target_pos']

        if controller in ['baseline_neural', 'wall_aware_neural']:

            desired, info = net.direction_to_goal(pos, heading, final_memory)

            control_source = 'goal_memory'

            effective_target = final_memory.position

        elif controller == 'waypoint_geometric_oracle':

            vec = target_pos - pos

            desired = vector_to_angle(vec) if np.linalg.norm(vec) > 1e-9 else heading

            info = {'goal_act': np.zeros(net.P.n_goal_dirs), 'v_from_goal_to_rat': np.zeros(2)}

            control_source = 'geometric_target'

            effective_target = target_pos

        else:

            desired_mem, info, mem = memory_direction_to_named_target(net, pos, heading, target_name, final_memory, waypoint_memories)

            vec_norm = float(np.linalg.norm(info['v_from_goal_to_rat']))

            use_recovery = False

            if controller == 'waypoint_memory_hybrid':

                if desired_mem is None or vec_norm < cfg.weak_vector_threshold or target_state['stall_count'] >= max(6, cfg.stall_patience_steps // 2):

                    use_recovery = True

            if desired_mem is None or use_recovery:

                vec = target_pos - pos

                desired = vector_to_angle(vec) if np.linalg.norm(vec) > 1e-9 else heading

                control_source = 'geometric_recovery' if use_recovery else 'geometric_missing_memory'

            else:

                desired = desired_mem

                control_source = 'waypoint_memory' if target_name != 'goal' else 'goal_memory'

            effective_target = target_pos

        smoothed = circular_mean_angle(heading, desired, wa=momentum, wb=1.0)

        if controller == 'baseline_neural':

            chosen_heading = smoothed

            new_pos = pos + STEP_CM * angle_to_vector(chosen_heading)

            blocked = not step_is_valid(pos, new_pos, walls, net.P)

            offset_deg = 0

            if blocked:

                blocked_count += 1

                new_pos = pos.copy()

                chosen_heading = wrap_angle(heading + net.rng.choice([-1, 1]) * np.pi/2)

        elif controller == 'wall_aware_neural':

            chosen_heading, new_pos, blocked, offset_deg = choose_progress_aware_heading(pos, smoothed, final_memory.position, STEP_CM, walls, net.P)

            if blocked:

                blocked_count += 1

        else:

            chosen_heading, new_pos, blocked, offset_deg = choose_progress_aware_heading(pos, smoothed, effective_target, STEP_CM, walls, net.P)

            if blocked:

                blocked_count += 1

        rows.append({

            'cycle': cycle,

            't_s': cycle / net.P.theta_hz,

            'x': pos[0], 'y': pos[1],

            'heading': heading,

            'desired': desired,

            'chosen_heading': chosen_heading,

            'heading_offset_deg': np.nan if offset_deg is None else float(offset_deg),

            'dist_to_goal': dist_to_goal,

            'target_name': target_name,

            'target_x': target_pos[0],

            'target_y': target_pos[1],

            'dist_to_target': float(np.linalg.norm(pos - target_pos)),

            'target_source': control_source,

            'path_labels': target_state['path_labels'],

            'path_estimate_cm': target_state['path_estimate_cm'],

            'target_switches': target_state['target_switches'],

            'stall_count': target_state['stall_count'],

            'forced_replans': target_state['forced_replans'],

            'lock_steps_left': target_state['lock_steps_left'],

            'goal_vector_norm': float(np.linalg.norm(info['v_from_goal_to_rat'])),

            'blocked': blocked,

            'blocked_count': blocked_count,

            **{f'GC_{label}': info['goal_act'][i] for i,(label,_) in enumerate(GOAL_DIRS)}

        })

        if dist_to_goal <= net.P.encounter_radius_cm:

            break

        heading = chosen_heading

        pos = new_pos

    return pd.DataFrame(rows)

def run_many_starts_v2(net: BurgessOkeefeNetwork, final_memory: GoalMemory, starts: List[np.ndarray],

                       waypoint_dict: Dict[str, np.ndarray], waypoint_memories: Optional[Dict[str, GoalMemory]],

                       controller: str, walls=None, max_seconds: float = 20.0,

                       cfg: TargetManagerConfig = TargetManagerConfig()):

    navs, rows = [], []

    for i, st in enumerate(starts):

        df = navigate_with_target_manager(net, st, final_memory, waypoint_dict, waypoint_memories,

                                          start_heading=net.rng.uniform(-np.pi, np.pi), max_seconds=max_seconds,

                                          walls=walls, controller=controller, cfg=cfg)

        df['run'] = i

        navs.append(df)

        rows.append({

            'run': i,

            'controller': controller,

            'success': bool(df.dist_to_goal.iloc[-1] <= net.P.encounter_radius_cm),

            'time_s': float(df.t_s.iloc[-1]),

            'final_dist_cm': float(df.dist_to_goal.iloc[-1]),

            'path_length_cm': path_length_cm(df),

            'blocked_steps': int(df.blocked.sum()),

            'target_switches': int(df.target_switches.iloc[-1]) if 'target_switches' in df else 0,

            'forced_replans': int(df.forced_replans.iloc[-1]) if 'forced_replans' in df else 0,

            'fraction_waypoint_mode': float((df.target_name != 'goal').mean()) if 'target_name' in df else 0.0,

            'fraction_geometric_recovery': float(df.target_source.astype(str).str.contains('geometric').mean()) if 'target_source' in df else 0.0,

        })

    return navs, pd.DataFrame(rows)

def plot_navs_with_waypoints(navs, goal, cues, waypoint_dict=None, title='Navigation'):

    fig, ax = plt.subplots(figsize=(6.5,6.5))

    plot_world(cues=cues, goal=goal, ax=ax, title=title)

    if waypoint_dict is not None:

        for name, w in waypoint_dict.items():

            w = np.asarray(w, dtype=float)

            if point_is_valid(w):

                ax.scatter([w[0]], [w[1]], marker='D', s=45, color='tab:purple')

                ax.text(w[0]+2, w[1]+2, name, fontsize=6, color='tab:purple')

    for df in navs:

        ax.plot(df.x, df.y, lw=1.45)

    return fig, ax

def save_v2_outputs(output_dir, comparison=None, navs_dict=None, cues=None, goal=None, waypoint_dict=None):

    out = Path(output_dir)

    out.mkdir(parents=True, exist_ok=True)

    if comparison is not None:

        comparison.to_csv(out / 'comparison_v2_models.csv', index=False)

    if navs_dict:

        for label, navs in navs_dict.items():

            d = out / f'navs_{label}'

            d.mkdir(exist_ok=True)

            for i, df in enumerate(navs):

                df.to_csv(d / f'run_{i:02d}.csv', index=False)

            fig, ax = plot_navs_with_waypoints(navs, goal, cues, waypoint_dict, label)

            fig.savefig(out / f'trajectories_{label}.png', dpi=180, bbox_inches='tight')

            plt.close(fig)

    if navs_dict and 'waypoint_geometric_oracle' in navs_dict and len(navs_dict['waypoint_geometric_oracle']) > 0:

        html = animate_navigation(navs_dict['waypoint_geometric_oracle'][0], goal, cues)

        (out / 'animation_waypoint_geometric_oracle_run0.html').write_text(html.data, encoding='utf-8')

    zip_path = out.with_suffix('.zip')

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:

        for fp in out.rglob('*'):

            zf.write(fp, fp.relative_to(out.parent))

    return str(zip_path)

def sample_valid_position(rng_obj=None, walls=None, P: Params = P, max_tries: int = 1000) -> np.ndarray:

    if rng_obj is None:

        rng_obj = rng

    if walls is None:

        walls = l_wall_segments_cm()

    for _ in range(max_tries):

        pos = rng_obj.uniform(P.agent_radius_cm, P.env_size_cm-P.agent_radius_cm, size=2)

        if point_is_valid(pos, walls, P, margin=True):

            return pos.astype(float)

    return np.array([P.env_size_cm*0.25, P.env_size_cm*0.25], dtype=float)

def _explore_environment_safe(self, seconds=30.0, start_pos=None, start_heading=None, walls=None):

    if walls is None:

        walls = l_wall_segments_cm()

    if start_pos is None:

        pos = sample_valid_position(self.rng, walls, self.P)

    else:

        pos = np.array(start_pos, dtype=float)

        if not point_is_valid(pos, walls, self.P, margin=True):

            pos = sample_valid_position(self.rng, walls, self.P)

    heading = float(start_heading if start_heading is not None else self.rng.uniform(-np.pi, np.pi))

    n_cycles = int(seconds * self.P.theta_hz)

    rows = []

    for cycle in range(n_cycles):

        total_updates_ec_pc = total_updates_pc_sc = 0

        for phase in PHASES:

            out = self.network_step(pos, heading, phase, learn=True)

            total_updates_ec_pc += out['updates_ec_pc']

            total_updates_pc_sc += out['updates_pc_sc']

        rows.append({'cycle':cycle, 't_s':cycle/self.P.theta_hz, 'x':pos[0], 'y':pos[1], 'heading':heading,

                     'updates_ec_pc':total_updates_ec_pc, 'updates_pc_sc':total_updates_pc_sc})

        moved = False

        for spread_deg in [self.P.exploration_turn_deg, 90, 180]:

            for _attempt in range(25):

                turn = self.rng.uniform(-np.deg2rad(spread_deg), np.deg2rad(spread_deg))

                h_try = wrap_angle(heading + turn)

                new_pos = pos + STEP_CM * angle_to_vector(h_try)

                if step_is_valid(pos, new_pos, walls, self.P):

                    heading = h_try

                    pos = new_pos

                    moved = True

                    break

            if moved:

                break

        if not moved:

            heading = self.rng.uniform(-np.pi, np.pi)

            pos = sample_valid_position(self.rng, walls, self.P)

    return pd.DataFrame(rows)

BurgessOkeefeNetwork.explore_environment = _explore_environment_safe

print('Professional v2 code loaded: target manager + hysteresis + progress-aware headings')

@dataclass

class ExperienceGraphConfig:

    merge_radius_cm: float = 9.0

    attach_radius_cm: float = 26.0

    min_edge_weight_cm: float = 1e-3

    use_line_of_sight_for_merge: bool = True

    node_feature_source: str = 'pc'                             

class ExperienceGraph:

    def __init__(self, walls=None, P: Params = P, config: ExperienceGraphConfig = ExperienceGraphConfig()):

        self.P = P

        self.walls = l_wall_segments_cm() if walls is None else walls

        self.cfg = config

        self.positions = []

        self.features = []

        self.counts = []

        self.edges = {}

    @property

    def n_nodes(self):

        return len(self.positions)

    def _new_node(self, pos, feature=None):

        idx = len(self.positions)

        self.positions.append(np.asarray(pos, dtype=float).copy())

        if feature is None:

            feature = np.asarray(pos, dtype=float)

        self.features.append(np.asarray(feature, dtype=float).copy())

        self.counts.append(1)

        self.edges[idx] = []

        return idx

    def _nearest_merge_candidate(self, pos):

        if self.n_nodes == 0:

            return None, float('inf')

        pts = np.vstack(self.positions)

        d = np.linalg.norm(pts - pos[None, :], axis=1)

        order = np.argsort(d)

        for idx in order[:12]:

            if d[idx] <= self.cfg.merge_radius_cm:

                if (not self.cfg.use_line_of_sight_for_merge) or path_is_clear(pos, self.positions[idx], self.walls, self.P, margin=True):

                    return int(idx), float(d[idx])

        return None, float('inf')

    def add_observation(self, pos, feature=None):

        pos = np.asarray(pos, dtype=float)

        idx, dist = self._nearest_merge_candidate(pos)

        if idx is None:

            return self._new_node(pos, feature)

        c = self.counts[idx]

        self.positions[idx] = (self.positions[idx] * c + pos) / (c + 1)

        if feature is not None:

            f = np.asarray(feature, dtype=float)

            self.features[idx] = (self.features[idx] * c + f) / (c + 1)

        self.counts[idx] = c + 1

        return idx

    def add_edge(self, a, b, weight=None):

        if a is None or b is None or a == b:

            return

        if weight is None:

            weight = float(np.linalg.norm(self.positions[a] - self.positions[b]))

        weight = max(float(weight), self.cfg.min_edge_weight_cm)

        for u, v in [(a,b), (b,a)]:

            old = self.edges.setdefault(u, [])

            for k, (vv, ww) in enumerate(old):

                if vv == v:

                    if weight < ww:

                        old[k] = (v, weight)

                    break

            else:

                old.append((v, weight))

    def nearest_visible_node(self, pos, max_radius=None):

        if self.n_nodes == 0:

            return None, float('inf')

        pos = np.asarray(pos, dtype=float)

        pts = np.vstack(self.positions)

        d = np.linalg.norm(pts - pos[None, :], axis=1)

        order = np.argsort(d)

        for idx in order:

            if max_radius is not None and d[idx] > max_radius:

                break

            if path_is_clear(pos, self.positions[int(idx)], self.walls, self.P, margin=True):

                return int(idx), float(d[idx])

        return int(order[0]), float(d[order[0]])

    def virtual_shortest_path(self, start_pos, goal_pos, attach_radius_cm=None):

        start_pos = np.asarray(start_pos, dtype=float)

        goal_pos = np.asarray(goal_pos, dtype=float)

        if attach_radius_cm is None:

            attach_radius_cm = self.cfg.attach_radius_cm

        if path_is_clear(start_pos, goal_pos, self.walls, self.P, margin=True):

            return ['current','goal'], [start_pos, goal_pos], float(np.linalg.norm(goal_pos - start_pos))

        if self.n_nodes == 0:

            return [], [], float('inf')

        base_nodes = [np.asarray(p, dtype=float) for p in self.positions]

        nodes = [start_pos] + base_nodes + [goal_pos]

        n = len(nodes)

        goal_idx = n - 1

        edges = {i: [] for i in range(n)}

        for u, lst in self.edges.items():

            for v, w in lst:

                edges[u+1].append((v+1, float(w)))

        pts = np.vstack(base_nodes)

        for virtual_idx, virtual_pos in [(0, start_pos), (goal_idx, goal_pos)]:

            d = np.linalg.norm(pts - virtual_pos[None, :], axis=1)

            order = np.argsort(d)

            added = 0

            for idx in order:

                if d[idx] > attach_radius_cm and added >= 2:

                    break

                node_pos = base_nodes[int(idx)]

                if path_is_clear(virtual_pos, node_pos, self.walls, self.P, margin=True):

                    real_idx = int(idx) + 1

                    w = float(d[idx])

                    edges[virtual_idx].append((real_idx, w))

                    edges[real_idx].append((virtual_idx, w))

                    added += 1

                    if added >= 6:

                        break

            if added == 0:

                nearest = int(order[0]) + 1

                w = float(d[order[0]]) * 1.5

                edges[virtual_idx].append((nearest, w))

                edges[nearest].append((virtual_idx, w))

        path_idx, total = _dijkstra(nodes, edges)

        if path_idx is None:

            return [], [], float('inf')

        labels = ['current']

        for pi in path_idx[1:-1]:

            labels.append(f'g{pi-1}')

        if len(path_idx) > 1:

            labels.append('goal')

        coords = [nodes[i] for i in path_idx]

        return labels, coords, total

    def to_dataframe(self):

        rows = []

        degrees = {i: len(self.edges.get(i, [])) for i in range(self.n_nodes)}

        for i, p in enumerate(self.positions):

            rows.append({'node': i, 'x': p[0], 'y': p[1], 'visits': self.counts[i], 'degree': degrees[i]})

        return pd.DataFrame(rows)

def phase_summed_pc_sc(net: BurgessOkeefeNetwork, pos, heading):

    pc_total = np.zeros(net.P.n_pc, dtype=np.float32)

    sc_total = np.zeros(net.P.n_sc, dtype=np.float32)

    for ph in PHASES:

        out = net.network_step(np.asarray(pos, dtype=float), float(heading), ph, learn=False)

        pc_total += out['pc']

        sc_total += out['sc']

    return pc_total, sc_total

def build_experience_graph_from_exploration(net: BurgessOkeefeNetwork, exploration_df: pd.DataFrame,

                                            walls=None, cfg: ExperienceGraphConfig = ExperienceGraphConfig(),

                                            every: int = 1) -> ExperienceGraph:

    if walls is None:

        walls = l_wall_segments_cm()

    graph = ExperienceGraph(walls=walls, P=net.P, config=cfg)

    prev_node = None

    prev_pos = None

    for row_idx, row in exploration_df.iloc[::max(1, every)].iterrows():

        pos = np.array([row.x, row.y], dtype=float)

        heading = float(row.heading)

        if cfg.node_feature_source == 'pc':

            pc, sc = phase_summed_pc_sc(net, pos, heading)

            feature = pc

        elif cfg.node_feature_source == 'sc':

            pc, sc = phase_summed_pc_sc(net, pos, heading)

            feature = sc

        else:

            feature = pos

        node = graph.add_observation(pos, feature)

        if prev_node is not None and node != prev_node:

            if prev_pos is not None and step_is_valid(prev_pos, pos, walls, net.P):

                graph.add_edge(prev_node, node, float(np.linalg.norm(pos - prev_pos)))

            elif path_is_clear(graph.positions[prev_node], graph.positions[node], walls, net.P, margin=True):

                graph.add_edge(prev_node, node, float(np.linalg.norm(graph.positions[node] - graph.positions[prev_node])))

        prev_node = node

        prev_pos = pos

    return graph

@dataclass

class GraphReplayConfig:

    subgoal_radius_cm: float = 13.0

    goal_radius_cm: float = P.encounter_radius_cm

    min_lock_steps: int = 8

    stall_patience_steps: int = 16

    min_progress_cm: float = 0.6

    weak_vector_threshold: float = 0.18

    direct_goal_when_visible: bool = True

class GraphReplayTargetManager:

    def __init__(self, graph: ExperienceGraph, goal_pos: np.ndarray, cfg: GraphReplayConfig = GraphReplayConfig(), walls=None, P: Params = P):

        self.graph = graph

        self.goal_pos = np.asarray(goal_pos, dtype=float)

        self.cfg = cfg

        self.walls = l_wall_segments_cm() if walls is None else walls

        self.P = P

        self.current_name = None

        self.current_pos = None

        self.path_labels = []

        self.path_estimate_cm = float('inf')

        self.lock_steps_left = 0

        self.best_dist = float('inf')

        self.stall_count = 0

        self.switch_count = 0

        self.forced_replans = 0

    def _radius(self, name):

        return self.cfg.goal_radius_cm if name == 'goal' else self.cfg.subgoal_radius_cm

    def _set_target(self, name, pos, labels, cost):

        old = self.current_name

        self.current_name = name

        self.current_pos = np.asarray(pos, dtype=float)

        self.path_labels = labels

        self.path_estimate_cm = float(cost)

        self.lock_steps_left = self.cfg.min_lock_steps

        self.best_dist = float('inf')

        self.stall_count = 0

        if old is not None and old != name:

            self.switch_count += 1

    def replan(self, pos):

        pos = np.asarray(pos, dtype=float)

        if self.cfg.direct_goal_when_visible and path_is_clear(pos, self.goal_pos, self.walls, self.P, margin=True):

            self._set_target('goal', self.goal_pos, ['current','goal'], float(np.linalg.norm(self.goal_pos-pos)))

            return

        labels, coords, total = self.graph.virtual_shortest_path(pos, self.goal_pos)

        if len(coords) >= 2:

            name = labels[1] if len(labels) > 1 else 'goal'

            tpos = coords[1]

            if name == 'goal':

                tpos = self.goal_pos

            self._set_target(name, tpos, labels, total)

        else:

            self._set_target('goal', self.goal_pos, ['current','goal_fallback'], float(np.linalg.norm(self.goal_pos-pos)))

    def update(self, pos, reliability: Optional[float] = None):

        pos = np.asarray(pos, dtype=float)

        if self.current_pos is None:

            self.replan(pos)

        dist = float(np.linalg.norm(pos - self.current_pos))

        if dist <= self._radius(self.current_name):

            self.replan(pos)

            dist = float(np.linalg.norm(pos - self.current_pos))

        if dist < self.best_dist - self.cfg.min_progress_cm:

            self.best_dist = dist

            self.stall_count = 0

        else:

            self.stall_count += 1

        if self.lock_steps_left > 0:

            self.lock_steps_left -= 1

        visible_goal = path_is_clear(pos, self.goal_pos, self.walls, self.P, margin=True)

        weak = reliability is not None and reliability < self.cfg.weak_vector_threshold

        if self.lock_steps_left <= 0:

            if visible_goal and self.current_name != 'goal':

                self.replan(pos)

            elif self.stall_count >= self.cfg.stall_patience_steps or weak:

                self.forced_replans += 1

                self.replan(pos)

        return {

            'target_name': self.current_name,

            'target_pos': self.current_pos.copy(),

            'dist_to_target': float(np.linalg.norm(pos - self.current_pos)),

            'path_labels': ' -> '.join(self.path_labels),

            'path_estimate_cm': self.path_estimate_cm,

            'target_switches': self.switch_count,

            'stall_count': self.stall_count,

            'forced_replans': self.forced_replans,

            'lock_steps_left': self.lock_steps_left,

        }

def goal_vector_reliability(info, eps: float = 1e-9):

    act = np.asarray(info.get('goal_act', []), dtype=float)

    v = np.asarray(info.get('v_from_goal_to_rat', [0.0, 0.0]), dtype=float)

    denom = float(np.sum(act)) + eps

    return float(np.linalg.norm(v) / denom)

def navigate_with_graph_replay(net: BurgessOkeefeNetwork, start_pos: np.ndarray, final_memory: GoalMemory,

                               graph: ExperienceGraph, start_heading: float = 0.0, max_seconds: float = 20.0,

                               momentum: float = 0.55, walls=None,

                               controller: str = 'pcgraph_geometric', cfg: GraphReplayConfig = GraphReplayConfig()) -> pd.DataFrame:

    if walls is None:

        walls = l_wall_segments_cm()

    if controller not in ['pcgraph_geometric', 'pcgraph_arbitrated']:

        raise ValueError(controller)

    pos = np.array(start_pos, dtype=float)

    heading = float(start_heading)

    manager = GraphReplayTargetManager(graph, final_memory.position, cfg=cfg, walls=walls, P=net.P)

    n_cycles = int(max_seconds * net.P.theta_hz)

    rows = []

    blocked_count = 0

    for cycle in range(n_cycles):

        dist_to_goal = float(np.linalg.norm(pos - final_memory.position))

        desired_goal, info = net.direction_to_goal(pos, heading, final_memory)

        reliability = goal_vector_reliability(info)

        state = manager.update(pos, reliability=reliability)

        target_pos = state['target_pos']

        target_name = state['target_name']

        if controller == 'pcgraph_geometric':

            vec = target_pos - pos

            desired = vector_to_angle(vec) if np.linalg.norm(vec) > 1e-9 else heading

            source = 'graph_subgoal_geometric' if target_name != 'goal' else 'goal_geometric_visible'

        else:

            visible_goal = path_is_clear(pos, final_memory.position, walls, net.P, margin=True)

            if visible_goal and reliability >= cfg.weak_vector_threshold:

                desired = desired_goal

                source = 'neural_goal_vector'

            else:

                vec = target_pos - pos

                desired = vector_to_angle(vec) if np.linalg.norm(vec) > 1e-9 else heading

                source = 'graph_subgoal_geometric'

        smoothed = circular_mean_angle(heading, desired, wa=momentum, wb=1.0)

        chosen_heading, new_pos, blocked, offset_deg = choose_progress_aware_heading(pos, smoothed, target_pos, STEP_CM, walls, net.P)

        if blocked:

            blocked_count += 1

        rows.append({

            'cycle': cycle, 't_s': cycle/net.P.theta_hz,

            'x': pos[0], 'y': pos[1], 'heading': heading,

            'desired': desired, 'chosen_heading': chosen_heading,

            'heading_offset_deg': np.nan if offset_deg is None else float(offset_deg),

            'dist_to_goal': dist_to_goal,

            'target_name': target_name,

            'target_x': target_pos[0], 'target_y': target_pos[1],

            'dist_to_target': float(np.linalg.norm(pos-target_pos)),

            'target_source': source,

            'path_labels': state['path_labels'],

            'path_estimate_cm': state['path_estimate_cm'],

            'target_switches': state['target_switches'],

            'stall_count': state['stall_count'],

            'forced_replans': state['forced_replans'],

            'lock_steps_left': state['lock_steps_left'],

            'goal_vector_norm': float(np.linalg.norm(info['v_from_goal_to_rat'])),

            'goal_vector_reliability': reliability,

            'blocked': blocked,

            'blocked_count': blocked_count,

            **{f'GC_{label}': info['goal_act'][i] for i,(label,_) in enumerate(GOAL_DIRS)}

        })

        if dist_to_goal <= net.P.encounter_radius_cm:

            break

        heading = chosen_heading

        pos = new_pos

    return pd.DataFrame(rows)

def path_length_from_points(points):

    pts = np.asarray(points, dtype=float)

    if len(pts) < 2:

        return 0.0

    return float(np.sum(np.linalg.norm(np.diff(pts, axis=0), axis=1)))

def shortest_visibility_path_length(start, goal, support_points: Optional[Dict[str, np.ndarray]] = None, walls=None, P: Params = P):

    if walls is None:

        walls = l_wall_segments_cm()

    if support_points is None:

        support_points = make_lwall_waypoints(P=P)

    labels, coords, total = visibility_graph_path(np.asarray(start), np.asarray(goal), support_points, walls, P)

    if np.isfinite(total):

        return float(total)

    return float(np.linalg.norm(np.asarray(goal)-np.asarray(start)))

def summarize_nav_dataframe(df, goal_pos, start_pos, optimal_len=None, P: Params = P):

    final_dist = float(df.dist_to_goal.iloc[-1]) if len(df) else float('inf')

    success = final_dist <= P.encounter_radius_cm

    length = path_length_cm(df) if 'x' in df else 0.0

    if optimal_len is None or optimal_len <= 1e-9 or not np.isfinite(optimal_len):

        spl = 0.0

        excess = np.nan

    else:

        spl = (1.0 if success else 0.0) * optimal_len / max(optimal_len, length)

        excess = length / optimal_len - 1.0

    return {

        'success': bool(success),

        'final_dist_cm': final_dist,

        'path_length_cm': length,

        'optimal_len_cm': optimal_len,

        'spl': float(spl),

        'excess_length': float(excess) if np.isfinite(excess) else np.nan,

        'blocked_steps': int(df.blocked.sum()) if 'blocked' in df else 0,

        'stall_final': int(df.stall_count.iloc[-1]) if 'stall_count' in df and len(df) else 0,

        'target_switches': int(df.target_switches.iloc[-1]) if 'target_switches' in df and len(df) else 0,

        'forced_replans': int(df.forced_replans.iloc[-1]) if 'forced_replans' in df and len(df) else 0,

        'steps': int(len(df)),

        'fraction_graph_mode': float((df.target_name != 'goal').mean()) if 'target_name' in df and len(df) else 0.0,

    }

def make_u_wall_segments_cm() -> List[np.ndarray]:

    return [

        np.array([[55.0, 35.0], [55.0, 115.0]], dtype=float),

        np.array([[55.0, 115.0], [115.0, 115.0]], dtype=float),

        np.array([[115.0, 115.0], [115.0, 55.0]], dtype=float),

    ]

def run_pcgraph_experiment(seed=0, arena='lwall', seconds_explore=45.0, max_seconds_nav=20.0,

                           cue_mode='perimeter_plus_wall', graph_every=1):

    walls = l_wall_segments_cm() if arena == 'lwall' else make_u_wall_segments_cm()

    cues = make_cues(cue_mode)

    net = BurgessOkeefeNetwork(cues, P=P, seed=seed)

    explore_df = net.explore_environment(seconds=seconds_explore, start_pos=np.array([25.0,25.0]), walls=walls)

    graph_cfg = ExperienceGraphConfig(merge_radius_cm=9.0, attach_radius_cm=28.0, node_feature_source='pc')

    graph = build_experience_graph_from_exploration(net, explore_df, walls=walls, cfg=graph_cfg, every=graph_every)

    goal_pos = np.array([105.0, 105.0]) if arena == 'lwall' else np.array([85.0, 82.0])

    final_memory = net.learn_goal('goal', goal_pos)

    starts = eight_edge_starts()

    rows = []

    navs = {}

    controllers = ['baseline_neural', 'wall_aware_neural', 'pcgraph_geometric', 'pcgraph_arbitrated']

    waypoint_dict = make_lwall_waypoints(P=P) if arena == 'lwall' else {}

    waypoint_memories = learn_waypoint_memories(net, waypoint_dict, walls=walls, P=P) if waypoint_dict else None

    if arena == 'lwall':

        controllers.append('waypoint_memory_hybrid')

    for controller in controllers:

        navs[controller] = []

        for si, st in enumerate(starts):

            h0 = net.rng.uniform(-np.pi, np.pi)

            if controller in ['baseline_neural','wall_aware_neural','waypoint_memory_hybrid']:

                if controller == 'waypoint_memory_hybrid':

                    df = navigate_with_target_manager(net, st, final_memory, waypoint_dict, waypoint_memories, start_heading=h0,

                                                      max_seconds=max_seconds_nav, walls=walls, controller=controller)

                else:

                    df = navigate_with_target_manager(net, st, final_memory, waypoint_dict, waypoint_memories, start_heading=h0,

                                                      max_seconds=max_seconds_nav, walls=walls, controller=controller)

            else:

                df = navigate_with_graph_replay(net, st, final_memory, graph, start_heading=h0, max_seconds=max_seconds_nav,

                                                walls=walls, controller=controller)

            navs[controller].append(df)

            opt = shortest_visibility_path_length(st, goal_pos, waypoint_dict if waypoint_dict else None, walls=walls, P=P)

            summary = summarize_nav_dataframe(df, goal_pos, st, optimal_len=opt, P=P)

            summary.update({'seed': seed, 'arena': arena, 'controller': controller, 'start_idx': si,

                            'graph_nodes': graph.n_nodes, 'graph_edges': sum(len(v) for v in graph.edges.values())//2})

            rows.append(summary)

    return {'net': net, 'graph': graph, 'explore_df': explore_df, 'goal': final_memory, 'navs': navs, 'summary': pd.DataFrame(rows), 'walls': walls, 'cues': cues}

def plot_graph_overlay(graph: ExperienceGraph, goal=None, starts=None, walls=None, ax=None, title='Experience graph'):

    if ax is None:

        fig, ax = plt.subplots(figsize=(6,6))

    if walls is None:

        walls = graph.walls

    ax.set_xlim(-5, P.env_size_cm+5); ax.set_ylim(-5, P.env_size_cm+5); ax.set_aspect('equal')

    ax.plot([0,P.env_size_cm,P.env_size_cm,0,0],[0,0,P.env_size_cm,P.env_size_cm,0], 'k-', lw=2)

    for w in walls:

        ax.plot(w[:,0], w[:,1], 'k-', lw=5, solid_capstyle='round')

    for u, lst in graph.edges.items():

        p = graph.positions[u]

        for v, wt in lst:

            if v > u:

                q = graph.positions[v]

                ax.plot([p[0],q[0]],[p[1],q[1]], '-', lw=0.5, alpha=0.35)

    if graph.n_nodes:

        pts = np.vstack(graph.positions)

        ax.scatter(pts[:,0], pts[:,1], s=12, alpha=0.8, label='experience nodes')

    goal_xy = _goal_to_xy(goal)

    if goal_xy is not None:

        ax.scatter([goal_xy[0]],[goal_xy[1]], marker='x', s=130, label='goal')

    if starts is not None:

        ss = np.asarray(starts)

        ax.scatter(ss[:,0], ss[:,1], marker='s', s=30, label='starts')

    ax.set_title(title); ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm'); ax.legend(loc='best')

    return ax

def save_pcgraph_outputs(output_dir, results_by_arena_seed):

    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)

    all_summaries = []

    for key, res in results_by_arena_seed.items():

        arena, seed = key

        sub = out / f'{arena}_seed{seed}'

        sub.mkdir(exist_ok=True)

        res['summary'].to_csv(sub / 'summary.csv', index=False)

        res['explore_df'].to_csv(sub / 'exploration.csv', index=False)

        res['graph'].to_dataframe().to_csv(sub / 'experience_graph_nodes.csv', index=False)

        all_summaries.append(res['summary'])

        fig, ax = plt.subplots(figsize=(6,6))

        plot_graph_overlay(res['graph'], res['goal'], starts=eight_edge_starts(), walls=res['walls'], ax=ax, title=f'{arena} seed={seed}: experience graph')

        fig.tight_layout(); fig.savefig(sub / 'experience_graph_overlay.png', dpi=180); plt.close(fig)

        for controller, navs in res['navs'].items():

            fig, ax = plt.subplots(figsize=(6,6))

            ax.set_xlim(-5,P.env_size_cm+5); ax.set_ylim(-5,P.env_size_cm+5); ax.set_aspect('equal')

            ax.plot([0,P.env_size_cm,P.env_size_cm,0,0],[0,0,P.env_size_cm,P.env_size_cm,0], 'k-', lw=2)

            for w in res['walls']:

                ax.plot(w[:,0], w[:,1], 'k-', lw=5, solid_capstyle='round')

            for i, df in enumerate(navs):

                ax.plot(df.x, df.y, lw=1.2, label=f'{i}' if i < 3 else None)

                ax.scatter([df.x.iloc[0]],[df.y.iloc[0]], s=20)

            g = res['goal'].position

            ax.scatter([g[0]],[g[1]], marker='x', s=130, label='goal')

            ax.set_title(f'{arena} seed={seed}: {controller}')

            ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm')

            fig.tight_layout(); fig.savefig(sub / f'trajectories_{controller}.png', dpi=180); plt.close(fig)

    combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()

    combined.to_csv(out / 'all_runs_summary.csv', index=False)

    if not combined.empty:

        agg = combined.groupby(['arena','controller']).agg(

            success_rate=('success','mean'),

            mean_spl=('spl','mean'),

            mean_excess=('excess_length','mean'),

            mean_path_length_cm=('path_length_cm','mean'),

            mean_blocked_steps=('blocked_steps','mean'),

            mean_target_switches=('target_switches','mean'),

            mean_forced_replans=('forced_replans','mean'),

            mean_fraction_graph_mode=('fraction_graph_mode','mean'),

            n=('success','size')

        ).reset_index()

        agg.to_csv(out / 'summary_by_arena_controller.csv', index=False)

        for metric in ['success_rate','mean_spl','mean_blocked_steps','mean_forced_replans','mean_fraction_graph_mode']:

            fig, ax = plt.subplots(figsize=(8,4))

            pivot = agg.pivot(index='controller', columns='arena', values=metric)

            pivot.plot(kind='bar', ax=ax)

            ax.set_title(metric); ax.set_ylim(0, max(1.0, np.nanmax(pivot.values)*1.15 if np.isfinite(pivot.values).any() else 1.0))

            fig.tight_layout(); fig.savefig(out / f'bar_{metric}.png', dpi=180); plt.close(fig)

    zip_path = out.with_suffix('.zip')

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:

        for fp in out.rglob('*'):

            zf.write(fp, fp.relative_to(out.parent))

    return str(zip_path)

@dataclass

class FeatureGraphConfig:

    merge_radius_cm: float = 10.0

    max_feature_merge_radius_cm: float = 18.0

    attach_radius_cm: float = 30.0

    feature_cosine_threshold: float = 0.72

    min_edge_weight_cm: float = 1e-3

    node_feature_source: str = 'pc'                                

    cost_aware_edges: bool = True

    wall_cost_scale: float = 10.0

    min_clearance_cm: float = 3.0

def cosine_similarity(a, b, eps=1e-9):

    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)

    na = float(np.linalg.norm(a)); nb = float(np.linalg.norm(b))

    if na < eps or nb < eps:

        return 0.0

    return float(np.dot(a, b) / (na * nb))

class PlaceBasisEncoder:

    def __init__(self, walls=None, P: Params = P, spacing_cm: float = 18.75, sigma_cm: float = 16.0):

        self.P = P

        self.walls = l_wall_segments_cm() if walls is None else walls

        self.sigma_cm = float(sigma_cm)

        coords = np.arange(spacing_cm, P.env_size_cm, spacing_cm)

        centers = []

        for x in coords:

            for y in coords:

                p = np.array([x, y], dtype=float)

                if point_is_valid(p, self.walls, P, margin=True):

                    centers.append(p)

        self.centers = np.asarray(centers, dtype=float)

    def encode(self, pos):

        pos = np.asarray(pos, dtype=float)

        if len(self.centers) == 0:

            return np.zeros(1, dtype=np.float32)

        d2 = np.sum((self.centers - pos[None, :])**2, axis=1)

        a = np.exp(-0.5 * d2 / (self.sigma_cm**2))

        return a.astype(np.float32)

class FeatureExperienceGraph:

    def __init__(self, walls=None, P: Params = P, config: FeatureGraphConfig = FeatureGraphConfig()):

        self.P = P

        self.walls = l_wall_segments_cm() if walls is None else walls

        self.cfg = config

        self.positions: List[np.ndarray] = []

        self.features: List[np.ndarray] = []

        self.counts: List[int] = []

        self.edges: Dict[int, List[Tuple[int, float]]] = {}

    @property

    def n_nodes(self):

        return len(self.positions)

    def _new_node(self, pos, feature):

        idx = self.n_nodes

        self.positions.append(np.asarray(pos, dtype=float).copy())

        self.features.append(np.asarray(feature, dtype=float).copy())

        self.counts.append(1)

        self.edges[idx] = []

        return idx

    def _candidate_by_position(self, pos):

        if self.n_nodes == 0:

            return None

        pts = np.vstack(self.positions)

        d = np.linalg.norm(pts - pos[None, :], axis=1)

        for idx in np.argsort(d)[:16]:

            if d[idx] <= self.cfg.merge_radius_cm and path_is_clear(pos, self.positions[int(idx)], self.walls, self.P, margin=True):

                return int(idx)

        return None

    def _candidate_by_feature(self, pos, feature):

        if self.n_nodes == 0 or self.cfg.node_feature_source == 'position':

            return None

        pts = np.vstack(self.positions)

        d = np.linalg.norm(pts - pos[None, :], axis=1)

        order = np.argsort(d)

        best = None; best_sim = -1.0

        for idx in order[:24]:

            idx = int(idx)

            if d[idx] > self.cfg.max_feature_merge_radius_cm:

                break

            if not path_is_clear(pos, self.positions[idx], self.walls, self.P, margin=True):

                continue

            sim = cosine_similarity(feature, self.features[idx])

            if sim >= self.cfg.feature_cosine_threshold and sim > best_sim:

                best = idx; best_sim = sim

        return best

    def add_observation(self, pos, feature):

        pos = np.asarray(pos, dtype=float)

        feature = np.asarray(feature, dtype=float)

        idx = self._candidate_by_position(pos)

        if idx is None:

            idx = self._candidate_by_feature(pos, feature)

        if idx is None:

            return self._new_node(pos, feature)

        c = self.counts[idx]

        self.positions[idx] = (self.positions[idx] * c + pos) / (c + 1)

        self.features[idx] = (self.features[idx] * c + feature) / (c + 1)

        self.counts[idx] = c + 1

        return idx

    def _edge_cost(self, a, b, prev=None):

        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)

        dist = float(np.linalg.norm(b - a))

        if not self.cfg.cost_aware_edges:

            return max(dist, self.cfg.min_edge_weight_cm)

        midpoint = 0.5*(a+b)

        clearance = min(point_segment_distance(midpoint, w[0], w[1]) for w in self.walls)

        wall_penalty = self.cfg.wall_cost_scale / max(self.cfg.min_clearance_cm, clearance)

        return max(dist * (1.0 + wall_penalty), self.cfg.min_edge_weight_cm)

    def add_edge(self, u, v):

        if u is None or v is None or u == v:

            return

        w = self._edge_cost(self.positions[u], self.positions[v])

        for a, b in [(u, v), (v, u)]:

            lst = self.edges.setdefault(a, [])

            for k, (bb, ww) in enumerate(lst):

                if bb == b:

                    if w < ww:

                        lst[k] = (b, w)

                    break

            else:

                lst.append((b, w))

    def virtual_shortest_path(self, start_pos, goal_pos, attach_radius_cm=None):

        start_pos = np.asarray(start_pos, dtype=float)

        goal_pos = np.asarray(goal_pos, dtype=float)

        if attach_radius_cm is None:

            attach_radius_cm = self.cfg.attach_radius_cm

        if path_is_clear(start_pos, goal_pos, self.walls, self.P, margin=True):

            return ['current', 'goal'], [start_pos, goal_pos], float(np.linalg.norm(goal_pos-start_pos))

        if self.n_nodes == 0:

            return [], [], float('inf')

        base_nodes = [np.asarray(p, dtype=float) for p in self.positions]

        nodes = [start_pos] + base_nodes + [goal_pos]

        n = len(nodes); goal_idx = n-1

        edges = {i: [] for i in range(n)}

        for u, lst in self.edges.items():

            for v, w in lst:

                edges[u+1].append((v+1, float(w)))

        pts = np.vstack(base_nodes)

        for virtual_idx, virtual_pos in [(0, start_pos), (goal_idx, goal_pos)]:

            d = np.linalg.norm(pts - virtual_pos[None, :], axis=1)

            added = 0

            for idx in np.argsort(d):

                if d[idx] > attach_radius_cm and added >= 2:

                    break

                node_pos = base_nodes[int(idx)]

                if path_is_clear(virtual_pos, node_pos, self.walls, self.P, margin=True):

                    real_idx = int(idx) + 1

                    w = self._edge_cost(virtual_pos, node_pos)

                    edges[virtual_idx].append((real_idx, w))

                    edges[real_idx].append((virtual_idx, w))

                    added += 1

                    if added >= 8:

                        break

            if added == 0:

                nearest = int(np.argsort(d)[0]) + 1

                w = float(d[nearest-1]) * 2.0

                edges[virtual_idx].append((nearest, w))

                edges[nearest].append((virtual_idx, w))

        path_idx, total = _dijkstra(nodes, edges)

        if path_idx is None:

            return [], [], float('inf')

        labels = ['current']

        for pi in path_idx[1:-1]:

            labels.append(f'g{pi-1}')

        if len(path_idx) > 1:

            labels.append('goal')

        return labels, [nodes[i] for i in path_idx], float(total)

    def to_dataframe(self):

        return pd.DataFrame([

            {'node': i, 'x': p[0], 'y': p[1], 'visits': self.counts[i], 'degree': len(self.edges.get(i, []))}

            for i, p in enumerate(self.positions)

        ])

def feature_for_graph(net, pos, heading, source, place_encoder=None):

    source = source.lower()

    if source == 'position':

        return np.asarray(pos, dtype=np.float32)

    if source in ['pc', 'sc']:

        pc, sc = phase_summed_pc_sc(net, pos, heading)

        return pc if source == 'pc' else sc

    if source == 'placebasis':

        if place_encoder is None:

            raise ValueError('place_encoder required for placebasis source')

        return place_encoder.encode(pos)

    raise ValueError(source)

def build_feature_experience_graph(net: BurgessOkeefeNetwork, exploration_df: pd.DataFrame,

                                   walls=None, cfg: FeatureGraphConfig = FeatureGraphConfig(), every: int = 1,

                                   place_encoder: Optional[PlaceBasisEncoder] = None) -> FeatureExperienceGraph:

    if walls is None:

        walls = l_wall_segments_cm()

    graph = FeatureExperienceGraph(walls=walls, P=net.P, config=cfg)

    prev_node = None

    prev_pos = None

    for _, row in exploration_df.iloc[::max(1, every)].iterrows():

        pos = np.array([row.x, row.y], dtype=float)

        heading = float(row.heading)

        feat = feature_for_graph(net, pos, heading, cfg.node_feature_source, place_encoder)

        node = graph.add_observation(pos, feat)

        if prev_node is not None and node != prev_node:

            if prev_pos is not None and step_is_valid(prev_pos, pos, walls, net.P):

                graph.add_edge(prev_node, node)

            elif path_is_clear(graph.positions[prev_node], graph.positions[node], walls, net.P, margin=True):

                graph.add_edge(prev_node, node)

        prev_node = node

        prev_pos = pos

    return graph

class FullHybridGraphReplayTargetManager(GraphReplayTargetManager):

    pass

def navigate_with_feature_graph_replay(net: BurgessOkeefeNetwork, start_pos: np.ndarray, final_memory: GoalMemory,

                                       graph: FeatureExperienceGraph, start_heading: float = 0.0,

                                       max_seconds: float = 20.0, momentum: float = 0.55, walls=None,

                                       controller: str = 'full_hybrid_arbitrated',

                                       cfg: GraphReplayConfig = GraphReplayConfig()) -> pd.DataFrame:

    if walls is None:

        walls = l_wall_segments_cm()

    if controller not in ['featuregraph_geometric', 'featuregraph_arbitrated', 'full_hybrid_arbitrated']:

        raise ValueError(controller)

    pos = np.array(start_pos, dtype=float)

    heading = float(start_heading)

    manager = GraphReplayTargetManager(graph, final_memory.position, cfg=cfg, walls=walls, P=net.P)

    n_cycles = int(max_seconds * net.P.theta_hz)

    rows = []

    blocked_count = 0

    for cycle in range(n_cycles):

        dist_to_goal = float(np.linalg.norm(pos - final_memory.position))

        desired_goal, info = net.direction_to_goal(pos, heading, final_memory)

        reliability = goal_vector_reliability(info)

        state = manager.update(pos, reliability=reliability)

        target_pos = state['target_pos']

        target_name = state['target_name']

        visible_goal = path_is_clear(pos, final_memory.position, walls, net.P, margin=True)

        if controller == 'featuregraph_geometric':

            use_neural = False

        elif controller == 'featuregraph_arbitrated':

            use_neural = visible_goal and reliability >= cfg.weak_vector_threshold

        else:

            use_neural = visible_goal and reliability >= cfg.weak_vector_threshold and state['stall_count'] < max(4, cfg.stall_patience_steps//3)

        if use_neural:

            desired = desired_goal

            effective_target = final_memory.position

            source = 'neural_goal_vector'

        else:

            vec = target_pos - pos

            desired = vector_to_angle(vec) if np.linalg.norm(vec) > 1e-9 else heading

            effective_target = target_pos

            source = 'feature_graph_subgoal' if target_name != 'goal' else 'goal_geometric'

        smoothed = circular_mean_angle(heading, desired, wa=momentum, wb=1.0)

        chosen_heading, new_pos, blocked, offset_deg = choose_progress_aware_heading(pos, smoothed, effective_target, STEP_CM, walls, net.P)

        if blocked:

            blocked_count += 1

        rows.append({

            'cycle': cycle, 't_s': cycle/net.P.theta_hz,

            'x': pos[0], 'y': pos[1], 'heading': heading,

            'desired': desired, 'chosen_heading': chosen_heading,

            'heading_offset_deg': np.nan if offset_deg is None else float(offset_deg),

            'dist_to_goal': dist_to_goal,

            'target_name': target_name,

            'target_x': target_pos[0], 'target_y': target_pos[1],

            'dist_to_target': float(np.linalg.norm(pos-target_pos)),

            'target_source': source,

            'path_labels': state['path_labels'],

            'path_estimate_cm': state['path_estimate_cm'],

            'target_switches': state['target_switches'],

            'stall_count': state['stall_count'],

            'forced_replans': state['forced_replans'],

            'lock_steps_left': state['lock_steps_left'],

            'goal_vector_norm': float(np.linalg.norm(info['v_from_goal_to_rat'])),

            'goal_vector_reliability': reliability,

            'blocked': blocked,

            'blocked_count': blocked_count,

            **{f'GC_{label}': info['goal_act'][i] for i,(label,_) in enumerate(GOAL_DIRS)}

        })

        if dist_to_goal <= net.P.encounter_radius_cm:

            break

        heading = chosen_heading

        pos = new_pos

    return pd.DataFrame(rows)

def make_wall_segments(arena: str):

    if arena == 'lwall':

        return l_wall_segments_cm()

    if arena == 'uwall':

        return make_u_wall_segments_cm()

    if arena == 'open':

        return []

    raise ValueError(arena)

def default_goal_for_arena(arena: str):

    if arena == 'lwall':

        return np.array([105.0, 105.0], dtype=float)

    if arena == 'uwall':

        return np.array([85.0, 82.0], dtype=float)

    if arena == 'open':

        return np.array([120.0, 120.0], dtype=float)

    raise ValueError(arena)

def shortest_path_reference(start, goal, walls=None, P: Params = P):

    if walls is None:

        walls = []

    if len(walls) == 0 or path_is_clear(start, goal, walls, P, margin=True):

        return float(np.linalg.norm(np.asarray(goal)-np.asarray(start)))

    support = {}

    clearance = 16.0

    k = 0

    for w in walls:

        for endpoint in [w[0], w[1]]:

            for dx in [-clearance, 0, clearance]:

                for dy in [-clearance, 0, clearance]:

                    if dx == 0 and dy == 0:

                        continue

                    p = endpoint + np.array([dx, dy], dtype=float)

                    if point_is_valid(p, walls, P, margin=True):

                        support[f'p{k}'] = p; k += 1

    labels, coords, total = visibility_graph_path(np.asarray(start), np.asarray(goal), support, walls, P)

    return float(total) if np.isfinite(total) else float(np.linalg.norm(np.asarray(goal)-np.asarray(start)))

def run_full_hybrid_experiment(seed=0, arena='lwall', seconds_explore=60.0, max_seconds_nav=22.0,

                               cue_mode='perimeter_plus_wall', graph_every=1,

                               feature_sources=('position', 'pc', 'sc', 'placebasis')):

    walls = make_wall_segments(arena)

    cues = make_cues(cue_mode)

    net = BurgessOkeefeNetwork(cues, P=P, seed=seed)

    start_explore = np.array([25.0, 25.0]) if arena != 'uwall' else np.array([25.0, 25.0])

    explore_df = net.explore_environment(seconds=seconds_explore, start_pos=start_explore, walls=walls)

    goal_pos = default_goal_for_arena(arena)

    final_memory = net.learn_goal('goal', goal_pos)

    starts = eight_edge_starts()

    graphs = {}

    for src in feature_sources:

        cfg = FeatureGraphConfig(node_feature_source=src, cost_aware_edges=True)

        enc = PlaceBasisEncoder(walls=walls, P=net.P) if src == 'placebasis' else None

        graphs[src] = build_feature_experience_graph(net, explore_df, walls=walls, cfg=cfg, every=graph_every, place_encoder=enc)

    waypoint_dict = make_lwall_waypoints(P=P) if arena == 'lwall' else {}

    waypoint_memories = learn_waypoint_memories(net, waypoint_dict, walls=walls, P=P) if waypoint_dict else None

    rows = []

    navs = {}

    base_controllers = ['baseline_neural', 'wall_aware_neural']

    if arena == 'lwall':

        base_controllers.append('waypoint_memory_hybrid')

    for controller in base_controllers:

        navs[controller] = []

        for si, st in enumerate(starts):

            h0 = net.rng.uniform(-np.pi, np.pi)

            df = navigate_with_target_manager(net, st, final_memory, waypoint_dict, waypoint_memories,

                                              start_heading=h0, max_seconds=max_seconds_nav,

                                              walls=walls, controller=controller)

            navs[controller].append(df)

            opt = shortest_path_reference(st, goal_pos, walls, P)

            summary = summarize_nav_dataframe(df, goal_pos, st, optimal_len=opt, P=P)

            summary.update({'seed': seed, 'arena': arena, 'controller': controller, 'graph_source': 'none',

                            'start_idx': si, 'graph_nodes': 0, 'graph_edges': 0})

            rows.append(summary)

    for src, graph in graphs.items():

        for controller in ['featuregraph_geometric', 'featuregraph_arbitrated', 'full_hybrid_arbitrated']:

            name = f'{src}_{controller}'

            navs[name] = []

            for si, st in enumerate(starts):

                h0 = net.rng.uniform(-np.pi, np.pi)

                df = navigate_with_feature_graph_replay(net, st, final_memory, graph,

                                                        start_heading=h0, max_seconds=max_seconds_nav,

                                                        walls=walls, controller=controller)

                navs[name].append(df)

                opt = shortest_path_reference(st, goal_pos, walls, P)

                summary = summarize_nav_dataframe(df, goal_pos, st, optimal_len=opt, P=P)

                summary.update({'seed': seed, 'arena': arena, 'controller': name, 'graph_source': src,

                                'start_idx': si, 'graph_nodes': graph.n_nodes,

                                'graph_edges': sum(len(v) for v in graph.edges.values())//2})

                rows.append(summary)

    return {'net': net, 'graphs': graphs, 'explore_df': explore_df, 'goal': final_memory,

            'navs': navs, 'summary': pd.DataFrame(rows), 'walls': walls, 'cues': cues, 'arena': arena, 'seed': seed}

def plot_feature_graph_overlay(graph: FeatureExperienceGraph, goal=None, starts=None, walls=None, ax=None, title='Feature experience graph'):

    if ax is None:

        fig, ax = plt.subplots(figsize=(6,6))

    if walls is None:

        walls = graph.walls

    ax.set_xlim(-5, P.env_size_cm+5); ax.set_ylim(-5, P.env_size_cm+5); ax.set_aspect('equal')

    ax.plot([0,P.env_size_cm,P.env_size_cm,0,0],[0,0,P.env_size_cm,P.env_size_cm,0], 'k-', lw=2)

    for w in walls:

        ax.plot(w[:,0], w[:,1], 'k-', lw=5, solid_capstyle='round')

    for u, lst in graph.edges.items():

        p = graph.positions[u]

        for v, wt in lst:

            if v > u:

                q = graph.positions[v]

                ax.plot([p[0], q[0]], [p[1], q[1]], '-', lw=0.45, alpha=0.32)

    if graph.n_nodes:

        pts = np.vstack(graph.positions)

        ax.scatter(pts[:,0], pts[:,1], s=10, alpha=0.85, label='nodes')

    goal_xy = _goal_to_xy(goal)

    if goal_xy is not None:

        ax.scatter([goal_xy[0]], [goal_xy[1]], marker='x', s=120, label='goal')

    if starts is not None:

        ss = np.asarray(starts)

        ax.scatter(ss[:,0], ss[:,1], marker='s', s=26, label='starts')

    ax.set_title(title); ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm')

    ax.legend(loc='best')

    return ax

def save_full_hybrid_outputs(output_dir, results_by_key):

    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)

    all_summaries = []

    for key, res in results_by_key.items():

        arena, seed = key

        sub = out / f'{arena}_seed{seed}'

        sub.mkdir(exist_ok=True)

        res['summary'].to_csv(sub / 'summary.csv', index=False)

        res['explore_df'].to_csv(sub / 'exploration.csv', index=False)

        all_summaries.append(res['summary'])

        for src, graph in res['graphs'].items():

            graph.to_dataframe().to_csv(sub / f'graph_nodes_{src}.csv', index=False)

            fig, ax = plt.subplots(figsize=(6,6))

            plot_feature_graph_overlay(graph, res['goal'], starts=eight_edge_starts(), walls=res['walls'], ax=ax, title=f'{arena} seed={seed}: {src} graph')

            fig.tight_layout(); fig.savefig(sub / f'graph_overlay_{src}.png', dpi=180); plt.close(fig)

        selected = [c for c in res['navs'].keys() if c in ['baseline_neural','wall_aware_neural','waypoint_memory_hybrid'] or c.endswith('full_hybrid_arbitrated')]

        for controller in selected:

            navs = res['navs'][controller]

            fig, ax = plt.subplots(figsize=(6,6))

            ax.set_xlim(-5,P.env_size_cm+5); ax.set_ylim(-5,P.env_size_cm+5); ax.set_aspect('equal')

            ax.plot([0,P.env_size_cm,P.env_size_cm,0,0],[0,0,P.env_size_cm,P.env_size_cm,0], 'k-', lw=2)

            for w in res['walls']:

                ax.plot(w[:,0], w[:,1], 'k-', lw=5, solid_capstyle='round')

            for i, df in enumerate(navs):

                ax.plot(df.x, df.y, lw=1.2)

                ax.scatter([df.x.iloc[0]],[df.y.iloc[0]], s=18)

            g = res['goal'].position

            ax.scatter([g[0]],[g[1]], marker='x', s=120, label='goal')

            ax.set_title(f'{arena} seed={seed}: {controller}')

            ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm')

            fig.tight_layout(); fig.savefig(sub / f'trajectories_{controller}.png', dpi=180); plt.close(fig)

    combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()

    combined.to_csv(out / 'all_runs_summary.csv', index=False)

    if not combined.empty:

        agg = combined.groupby(['arena','controller','graph_source']).agg(

            success_rate=('success','mean'),

            mean_spl=('spl','mean'),

            mean_excess=('excess_length','mean'),

            mean_path_length_cm=('path_length_cm','mean'),

            mean_blocked_steps=('blocked_steps','mean'),

            mean_target_switches=('target_switches','mean'),

            mean_forced_replans=('forced_replans','mean'),

            mean_fraction_graph_mode=('fraction_graph_mode','mean'),

            mean_graph_nodes=('graph_nodes','mean'),

            n=('success','size')

        ).reset_index()

        agg.to_csv(out / 'summary_by_arena_controller.csv', index=False)

        for metric in ['success_rate','mean_spl','mean_blocked_steps','mean_forced_replans','mean_fraction_graph_mode']:

            fig, ax = plt.subplots(figsize=(11,5))

            agg_plot = agg.copy()

            agg_plot['method'] = agg_plot['controller'].str.replace('_featuregraph_', '_fg_', regex=False)

            pivot = agg_plot.pivot_table(index='method', columns='arena', values=metric, aggfunc='mean')

            pivot.plot(kind='bar', ax=ax)

            ax.set_title(metric); ax.set_ylabel(metric)

            if metric in ['success_rate','mean_spl','mean_fraction_graph_mode']:

                ax.set_ylim(0, 1.05)

            fig.tight_layout(); fig.savefig(out / f'bar_{metric}.png', dpi=180); plt.close(fig)

    zip_path = out.with_suffix('.zip')

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:

        for fp in out.rglob('*'):

            zf.write(fp, fp.relative_to(out.parent))

    return str(zip_path)

print('Full hybrid extension loaded: PC/SC/place-basis experience graphs + replay arbitration')

def choose_progress_aware_heading(pos: np.ndarray, desired_heading: float, target_pos: np.ndarray,

                                  step_cm: float = STEP_CM, walls=None, P: Params = P,

                                  max_offset_deg: int = 180):

    if walls is None:

        walls = l_wall_segments_cm()

    offsets = [0]

    for d in [5,10,15,20,30,40,50,60,75,90,120,150,180]:

        offsets.extend([d, -d])

    best = None

    target_pos = np.asarray(target_pos, dtype=float)

    for off in offsets:

        if abs(off) > max_offset_deg:

            continue

        h = wrap_angle(desired_heading + np.deg2rad(off))

        new_pos = pos + step_cm * angle_to_vector(h)

        if not step_is_valid(pos, new_pos, walls, P):

            continue

        progress_cost = float(np.linalg.norm(new_pos - target_pos))

        angular_cost = 0.04 * abs(off)

        if len(walls) > 0:

            wall_clearance = min(point_segment_distance(new_pos, w[0], w[1]) for w in walls)

            clearance_penalty = 1.5 / max(1.0, wall_clearance)

        else:

            clearance_penalty = 0.0

        score = progress_cost + angular_cost + clearance_penalty

        if best is None or score < best[0]:

            best = (score, h, new_pos, off)

    if best is None:

        return desired_heading, pos.copy(), True, None

    return best[1], best[2], False, best[3]

def _feature_edge_cost_patched(self, a, b, prev=None):

    a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)

    dist = float(np.linalg.norm(b - a))

    if (not self.cfg.cost_aware_edges) or len(self.walls) == 0:

        return max(dist, self.cfg.min_edge_weight_cm)

    midpoint = 0.5*(a+b)

    clearance = min(point_segment_distance(midpoint, w[0], w[1]) for w in self.walls)

    wall_penalty = self.cfg.wall_cost_scale / max(self.cfg.min_clearance_cm, clearance)

    return max(dist * (1.0 + wall_penalty), self.cfg.min_edge_weight_cm)

FeatureExperienceGraph._edge_cost = _feature_edge_cost_patched

print('Open-arena patches applied')



In [ ]:

@dataclass

class TopologyGraphConfig:

    merge_radius_cm: float = 8.0

    attach_radius_cm: float = 32.0

    feature_cosine_threshold: float = 0.35

    node_feature_source: str = 'position'                                

    cost_aware_edges: bool = True

    wall_cost_scale: float = 8.0

    min_clearance_cm: float = 4.0

    min_edge_weight_cm: float = 1e-3

class TopologyPreservingFeatureGraph:

    def __init__(self, walls=None, P: Params = P, config: TopologyGraphConfig = TopologyGraphConfig()):

        self.P = P

        self.walls = [] if walls is None else walls

        self.cfg = config

        self.positions = []

        self.features = []

        self.counts = []

        self.edges = {}

        self.edge_visits = {}

    @property

    def n_nodes(self):

        return len(self.positions)

    def _new_node(self, pos, feature):

        idx = self.n_nodes

        self.positions.append(np.asarray(pos, dtype=float).copy())

        self.features.append(np.asarray(feature, dtype=float).copy())

        self.counts.append(1)

        self.edges[idx] = []

        return idx

    def _candidate(self, pos, feature):

        if self.n_nodes == 0:

            return None

        pos = np.asarray(pos, dtype=float)

        pts = np.vstack(self.positions)

        d = np.linalg.norm(pts - pos[None, :], axis=1)

        best = None

        best_score = float('inf')

        for idx in np.argsort(d)[:24]:

            idx = int(idx)

            if d[idx] > self.cfg.merge_radius_cm:

                break

            if not path_is_clear(pos, self.positions[idx], self.walls, self.P, margin=True):

                continue

            if self.cfg.node_feature_source != 'position':

                sim = cosine_similarity(feature, self.features[idx])

                if sim < self.cfg.feature_cosine_threshold:

                    continue

                score = float(d[idx]) - 0.2 * sim

            else:

                score = float(d[idx])

            if score < best_score:

                best = idx

                best_score = score

        return best

    def add_observation(self, pos, feature):

        pos = np.asarray(pos, dtype=float)

        feature = np.asarray(feature, dtype=float)

        idx = self._candidate(pos, feature)

        if idx is None:

            return self._new_node(pos, feature)

        c = self.counts[idx]

        self.positions[idx] = (self.positions[idx] * c + pos) / (c + 1)

        self.features[idx] = (self.features[idx] * c + feature) / (c + 1)

        self.counts[idx] = c + 1

        return idx

    def _edge_cost(self, a, b):

        a = np.asarray(a, dtype=float); b = np.asarray(b, dtype=float)

        dist = float(np.linalg.norm(b - a))

        if len(self.walls) == 0 or not self.cfg.cost_aware_edges:

            return max(dist, self.cfg.min_edge_weight_cm)

        midpoint = 0.5*(a+b)

        clearance = min(point_segment_distance(midpoint, w[0], w[1]) for w in self.walls)

        wall_penalty = self.cfg.wall_cost_scale / max(self.cfg.min_clearance_cm, clearance)

        return max(dist * (1.0 + wall_penalty), self.cfg.min_edge_weight_cm)

    def add_edge(self, u, v):

        if u is None or v is None or u == v:

            return

        if not path_is_clear(self.positions[u], self.positions[v], self.walls, self.P, margin=True):

            return

        key = tuple(sorted((int(u), int(v))))

        self.edge_visits[key] = self.edge_visits.get(key, 0) + 1

        base_cost = self._edge_cost(self.positions[u], self.positions[v])

        cost = base_cost / math.sqrt(self.edge_visits[key])

        for a, b in [(u, v), (v, u)]:

            lst = self.edges.setdefault(a, [])

            for k, (bb, ww) in enumerate(lst):

                if bb == b:

                    if cost < ww:

                        lst[k] = (b, cost)

                    break

            else:

                lst.append((b, cost))

    def _attach_virtual(self, edges, nodes, virtual_idx, virtual_pos, base_offset=1):

        if self.n_nodes == 0:

            return 0

        pts = np.vstack(self.positions)

        d = np.linalg.norm(pts - virtual_pos[None, :], axis=1)

        added = 0

        for idx in np.argsort(d):

            idx = int(idx)

            if d[idx] > self.cfg.attach_radius_cm and added >= 2:

                break

            if path_is_clear(virtual_pos, self.positions[idx], self.walls, self.P, margin=True):

                real_idx = idx + base_offset

                w = self._edge_cost(virtual_pos, self.positions[idx])

                edges[virtual_idx].append((real_idx, w))

                edges[real_idx].append((virtual_idx, w))

                added += 1

                if added >= 8:

                    break

        if added == 0:

            nearest = int(np.argsort(d)[0]) + base_offset

            w = float(d[nearest-base_offset]) * 2.0

            edges[virtual_idx].append((nearest, w))

            edges[nearest].append((virtual_idx, w))

            added = 1

        return added

    def virtual_shortest_path(self, start_pos, goal_pos, attach_radius_cm=None):

        start_pos = np.asarray(start_pos, dtype=float)

        goal_pos = np.asarray(goal_pos, dtype=float)

        if path_is_clear(start_pos, goal_pos, self.walls, self.P, margin=True):

            return ['current', 'goal'], [start_pos, goal_pos], float(np.linalg.norm(goal_pos-start_pos))

        if self.n_nodes == 0:

            return [], [], float('inf')

        base_nodes = [np.asarray(p, dtype=float) for p in self.positions]

        nodes = [start_pos] + base_nodes + [goal_pos]

        n = len(nodes); goal_idx = n-1

        edges = {i: [] for i in range(n)}

        for u, lst in self.edges.items():

            for v, w in lst:

                edges[u+1].append((v+1, float(w)))

        old_attach = self.cfg.attach_radius_cm

        if attach_radius_cm is not None:

            self.cfg.attach_radius_cm = attach_radius_cm

        try:

            self._attach_virtual(edges, nodes, 0, start_pos)

            self._attach_virtual(edges, nodes, goal_idx, goal_pos)

        finally:

            self.cfg.attach_radius_cm = old_attach

        path_idx, total = _dijkstra(nodes, edges)

        if path_idx is None:

            return [], [], float('inf')

        labels = ['current']

        for pi in path_idx[1:-1]:

            labels.append(f'g{pi-1}')

        if len(path_idx) > 1:

            labels.append('goal')

        return labels, [nodes[i] for i in path_idx], float(total)

    def wavefront_values(self, goal_pos):

        if self.n_nodes == 0:

            return np.array([])

        goal_pos = np.asarray(goal_pos, dtype=float)

        n = self.n_nodes + 1

        goal_idx = self.n_nodes

        edges = {i: [] for i in range(n)}

        for u, lst in self.edges.items():

            for v, w in lst:

                edges[u].append((v, float(w)))

        pts = np.vstack(self.positions)

        d = np.linalg.norm(pts - goal_pos[None, :], axis=1)

        added = 0

        for idx in np.argsort(d):

            idx = int(idx)

            if d[idx] > self.cfg.attach_radius_cm and added >= 2:

                break

            if path_is_clear(goal_pos, self.positions[idx], self.walls, self.P, margin=True):

                w = self._edge_cost(goal_pos, self.positions[idx])

                edges[goal_idx].append((idx, w)); edges[idx].append((goal_idx, w))

                added += 1

                if added >= 8:

                    break

        import heapq

        dist = np.full(n, np.inf)

        dist[goal_idx] = 0.0

        pq = [(0.0, goal_idx)]

        while pq:

            du, u = heapq.heappop(pq)

            if du != dist[u]:

                continue

            for v, w in edges.get(u, []):

                nd = du + w

                if nd < dist[v]:

                    dist[v] = nd

                    heapq.heappush(pq, (nd, v))

        return dist[:self.n_nodes]

    def to_dataframe(self, goal_pos=None):

        vals = self.wavefront_values(goal_pos) if goal_pos is not None else None

        rows = []

        for i, p in enumerate(self.positions):

            row = {'node': i, 'x': p[0], 'y': p[1], 'visits': self.counts[i],

                   'degree': len(self.edges.get(i, []))}

            if vals is not None and len(vals):

                row['wavefront_value'] = vals[i]

            rows.append(row)

        return pd.DataFrame(rows)

def make_s_corridor_segments_cm():

    return [

        np.array([[45.0, 0.0], [45.0, 105.0]], dtype=float),

        np.array([[105.0, 45.0], [105.0, 150.0]], dtype=float),

    ]

def make_wall_segments_v2(arena: str):

    if arena == 'lwall':

        return l_wall_segments_cm()

    if arena == 'uwall':

        return make_u_wall_segments_cm()

    if arena == 'scorridor':

        return make_s_corridor_segments_cm()

    if arena == 'open':

        return []

    raise ValueError(arena)

def default_goal_for_arena_v2(arena: str):

    if arena == 'lwall':

        return np.array([105.0, 105.0], dtype=float)

    if arena == 'uwall':

        return np.array([85.0, 82.0], dtype=float)

    if arena == 'scorridor':

        return np.array([130.0, 130.0], dtype=float)

    if arena == 'open':

        return np.array([120.0, 120.0], dtype=float)

    raise ValueError(arena)

def starts_for_arena_v2(arena: str):

    if arena == 'scorridor':

        return np.array([

            [15, 15], [20, 55], [20, 95], [25, 135],

            [75, 20], [75, 130], [135, 25], [135, 75]

        ], dtype=float)

    return eight_edge_starts()

def build_topology_graph(net: BurgessOkeefeNetwork, exploration_df: pd.DataFrame, walls, source='pc', every=1,

                         merge_radius_cm=8.0, feature_threshold=None):

    if feature_threshold is None:

        feature_threshold = {

            'position': 0.0,

            'pc': 0.28,

            'sc': 0.28,

            'placebasis': 0.55,

        }.get(source, 0.35)

    cfg = TopologyGraphConfig(

        merge_radius_cm=merge_radius_cm,

        attach_radius_cm=34.0,

        feature_cosine_threshold=feature_threshold,

        node_feature_source=source,

        cost_aware_edges=True,

        wall_cost_scale=8.0,

    )

    graph = TopologyPreservingFeatureGraph(walls=walls, P=net.P, config=cfg)

    enc = PlaceBasisEncoder(walls=walls, P=net.P, spacing_cm=15.0, sigma_cm=15.0) if source == 'placebasis' else None

    prev_node = None; prev_pos = None

    for _, row in exploration_df.iloc[::max(1, every)].iterrows():

        pos = np.array([row.x, row.y], dtype=float)

        heading = float(row.heading)

        feat = feature_for_graph(net, pos, heading, source, enc)

        node = graph.add_observation(pos, feat)

        if prev_node is not None and node != prev_node:

            if prev_pos is not None and step_is_valid(prev_pos, pos, walls, net.P):

                graph.add_edge(prev_node, node)

            elif path_is_clear(graph.positions[prev_node], graph.positions[node], walls, net.P, margin=True):

                graph.add_edge(prev_node, node)

        prev_node = node; prev_pos = pos

    return graph

def navigate_with_topology_graph(net: BurgessOkeefeNetwork, start_pos: np.ndarray, final_memory: GoalMemory,

                                 graph: TopologyPreservingFeatureGraph, start_heading: float = 0.0,

                                 max_seconds: float = 24.0, momentum: float = 0.55, walls=None,

                                 controller: str = 'topograph_arbitrated',

                                 cfg: GraphReplayConfig = GraphReplayConfig()) -> pd.DataFrame:

    if walls is None:

        walls = []

    if controller not in ['topograph_geometric', 'topograph_arbitrated', 'topograph_full_hybrid']:

        raise ValueError(controller)

    pos = np.array(start_pos, dtype=float)

    heading = float(start_heading)

    manager = GraphReplayTargetManager(graph, final_memory.position, cfg=cfg, walls=walls, P=net.P)

    n_cycles = int(max_seconds * net.P.theta_hz)

    rows = []

    blocked_count = 0

    for cycle in range(n_cycles):

        dist_to_goal = float(np.linalg.norm(pos - final_memory.position))

        desired_goal, info = net.direction_to_goal(pos, heading, final_memory)

        reliability = goal_vector_reliability(info)

        state = manager.update(pos, reliability=reliability)

        target_pos = state['target_pos']

        target_name = state['target_name']

        visible_goal = path_is_clear(pos, final_memory.position, walls, net.P, margin=True)

        blocked_direct = not step_is_valid(pos, pos + STEP_CM * angle_to_vector(desired_goal), walls, net.P)

        if controller == 'topograph_geometric':

            use_neural = False

        elif controller == 'topograph_arbitrated':

            use_neural = visible_goal and reliability >= cfg.weak_vector_threshold and not blocked_direct

        else:

            use_neural = (visible_goal and reliability >= cfg.weak_vector_threshold and

                          state['stall_count'] < max(3, cfg.stall_patience_steps//4) and not blocked_direct)

        if use_neural:

            desired = desired_goal

            effective_target = final_memory.position

            source = 'neural_goal_vector'

        else:

            vec = target_pos - pos

            desired = vector_to_angle(vec) if np.linalg.norm(vec) > 1e-9 else heading

            effective_target = target_pos

            source = 'topological_replay_subgoal' if target_name != 'goal' else 'goal_geometric'

        smoothed = circular_mean_angle(heading, desired, wa=momentum, wb=1.0)

        chosen_heading, new_pos, blocked, offset_deg = choose_progress_aware_heading(pos, smoothed, effective_target, STEP_CM, walls, net.P)

        if blocked:

            blocked_count += 1

        rows.append({

            'cycle': cycle, 't_s': cycle/net.P.theta_hz,

            'x': pos[0], 'y': pos[1], 'heading': heading,

            'desired': desired, 'chosen_heading': chosen_heading,

            'heading_offset_deg': np.nan if offset_deg is None else float(offset_deg),

            'dist_to_goal': dist_to_goal,

            'target_name': target_name,

            'target_x': target_pos[0], 'target_y': target_pos[1],

            'dist_to_target': float(np.linalg.norm(pos-target_pos)),

            'target_source': source,

            'path_labels': state['path_labels'],

            'path_estimate_cm': state['path_estimate_cm'],

            'target_switches': state['target_switches'],

            'stall_count': state['stall_count'],

            'forced_replans': state['forced_replans'],

            'lock_steps_left': state['lock_steps_left'],

            'goal_vector_norm': float(np.linalg.norm(info['v_from_goal_to_rat'])),

            'goal_vector_reliability': reliability,

            'blocked': blocked,

            'blocked_count': blocked_count,

            'blocked_direct_neural_step': blocked_direct,

            **{f'GC_{label}': info['goal_act'][i] for i,(label,_) in enumerate(GOAL_DIRS)}

        })

        if dist_to_goal <= net.P.encounter_radius_cm:

            break

        heading = chosen_heading

        pos = new_pos

    return pd.DataFrame(rows)

def run__graph_v2(seed=0, arena='uwall', seconds_explore=70.0, max_seconds_nav=26.0,

                               cue_mode='perimeter_plus_wall', graph_every=1,

                               graph_sources=('position','pc','sc','placebasis')):

    walls = make_wall_segments_v2(arena)

    cues = make_cues(cue_mode)

    net = BurgessOkeefeNetwork(cues, P=P, seed=seed)

    start_explore = np.array([25.0, 25.0], dtype=float)

    explore_df = net.explore_environment(seconds=seconds_explore, start_pos=start_explore, walls=walls)

    goal_pos = default_goal_for_arena_v2(arena)

    final_memory = net.learn_goal('goal', goal_pos)

    starts = starts_for_arena_v2(arena)

    graphs = {}

    for src in graph_sources:

        graphs[src] = build_topology_graph(net, explore_df, walls, source=src, every=graph_every)

    waypoint_dict = make_lwall_waypoints(P=P) if arena == 'lwall' else {}

    waypoint_memories = learn_waypoint_memories(net, waypoint_dict, walls=walls, P=P) if waypoint_dict else None

    rows = []

    navs = {}

    base_controllers = ['baseline_neural','wall_aware_neural']

    if arena == 'lwall':

        base_controllers.append('waypoint_memory_hybrid')

    for controller in base_controllers:

        navs[controller] = []

        for si, st in enumerate(starts):

            h0 = net.rng.uniform(-np.pi, np.pi)

            df = navigate_with_target_manager(net, st, final_memory, waypoint_dict, waypoint_memories,

                                              start_heading=h0, max_seconds=max_seconds_nav,

                                              walls=walls, controller=controller)

            navs[controller].append(df)

            opt = shortest_path_reference(st, goal_pos, walls, P)

            summary = summarize_nav_dataframe(df, goal_pos, st, optimal_len=opt, P=P)

            summary.update({'seed': seed, 'arena': arena, 'controller': controller, 'graph_source': 'none',

                            'start_idx': si, 'graph_nodes': 0, 'graph_edges': 0})

            rows.append(summary)

    for src, graph in graphs.items():

        for controller in ['topograph_geometric', 'topograph_arbitrated', 'topograph_full_hybrid']:

            name = f'{src}_{controller}'

            navs[name] = []

            for si, st in enumerate(starts):

                h0 = net.rng.uniform(-np.pi, np.pi)

                df = navigate_with_topology_graph(net, st, final_memory, graph,

                                                  start_heading=h0, max_seconds=max_seconds_nav,

                                                  walls=walls, controller=controller)

                navs[name].append(df)

                opt = shortest_path_reference(st, goal_pos, walls, P)

                summary = summarize_nav_dataframe(df, goal_pos, st, optimal_len=opt, P=P)

                summary.update({'seed': seed, 'arena': arena, 'controller': name, 'graph_source': src,

                                'start_idx': si, 'graph_nodes': graph.n_nodes,

                                'graph_edges': sum(len(v) for v in graph.edges.values())//2})

                if len(df):

                    summary['mean_goal_reliability'] = float(df.goal_vector_reliability.mean())

                    summary['direct_neural_block_ratio'] = float(df.blocked_direct_neural_step.mean()) if 'blocked_direct_neural_step' in df else np.nan

                rows.append(summary)

    return {'net': net, 'graphs': graphs, 'explore_df': explore_df, 'goal': final_memory,

            'navs': navs, 'summary': pd.DataFrame(rows), 'walls': walls, 'cues': cues,

            'arena': arena, 'seed': seed}

def plot_topology_graph_overlay(graph, goal=None, starts=None, walls=None, ax=None, title='Topology-preserving graph'):

    if ax is None:

        fig, ax = plt.subplots(figsize=(6,6))

    if walls is None:

        walls = graph.walls

    ax.set_xlim(-5, P.env_size_cm+5); ax.set_ylim(-5, P.env_size_cm+5); ax.set_aspect('equal')

    ax.plot([0,P.env_size_cm,P.env_size_cm,0,0],[0,0,P.env_size_cm,P.env_size_cm,0], 'k-', lw=2)

    for w in walls:

        ax.plot(w[:,0], w[:,1], 'k-', lw=5, solid_capstyle='round')

    for u, lst in graph.edges.items():

        p = graph.positions[u]

        for v, wt in lst:

            if v > u:

                q = graph.positions[v]

                ax.plot([p[0], q[0]], [p[1], q[1]], '-', lw=0.45, alpha=0.35)

    if graph.n_nodes:

        pts = np.vstack(graph.positions)

        ax.scatter(pts[:,0], pts[:,1], s=10, alpha=0.88, label='nodes')

    goal_xy = _goal_to_xy(goal)

    if goal_xy is not None:

        ax.scatter([goal_xy[0]],[goal_xy[1]], marker='x', s=120, label='goal')

    if starts is not None:

        ss = np.asarray(starts)

        ax.scatter(ss[:,0], ss[:,1], marker='s', s=26, label='starts')

    ax.set_title(title); ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm')

    ax.legend(loc='best')

    return ax

def save__v2_outputs(output_dir, results_by_key):

    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)

    all_summaries = []

    for key, res in results_by_key.items():

        arena, seed = key

        sub = out / f'{arena}_seed{seed}'

        sub.mkdir(exist_ok=True)

        res['summary'].to_csv(sub / 'summary.csv', index=False)

        res['explore_df'].to_csv(sub / 'exploration.csv', index=False)

        all_summaries.append(res['summary'])

        for src, graph in res['graphs'].items():

            graph.to_dataframe(goal_pos=res['goal'].position).to_csv(sub / f'topology_graph_nodes_{src}.csv', index=False)

            fig, ax = plt.subplots(figsize=(6,6))

            plot_topology_graph_overlay(graph, res['goal'], starts=starts_for_arena_v2(arena), walls=res['walls'], ax=ax,

                                        title=f'{arena} seed={seed}: {src} topology graph')

            fig.tight_layout(); fig.savefig(sub / f'topology_graph_overlay_{src}.png', dpi=180); plt.close(fig)

        selected = [c for c in res['navs'].keys() if c in ['baseline_neural','wall_aware_neural','waypoint_memory_hybrid'] or c.endswith('topograph_full_hybrid')]

        for controller in selected:

            navs = res['navs'][controller]

            fig, ax = plt.subplots(figsize=(6,6))

            ax.set_xlim(-5,P.env_size_cm+5); ax.set_ylim(-5,P.env_size_cm+5); ax.set_aspect('equal')

            ax.plot([0,P.env_size_cm,P.env_size_cm,0,0],[0,0,P.env_size_cm,P.env_size_cm,0], 'k-', lw=2)

            for w in res['walls']:

                ax.plot(w[:,0], w[:,1], 'k-', lw=5, solid_capstyle='round')

            for i, df in enumerate(navs):

                ax.plot(df.x, df.y, lw=1.2)

                ax.scatter([df.x.iloc[0]],[df.y.iloc[0]], s=18)

            g = res['goal'].position

            ax.scatter([g[0]],[g[1]], marker='x', s=120, label='goal')

            ax.set_title(f'{arena} seed={seed}: {controller}')

            ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm')

            fig.tight_layout(); fig.savefig(sub / f'trajectories_{controller}.png', dpi=180); plt.close(fig)

    combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()

    combined.to_csv(out / 'all_runs_summary.csv', index=False)

    if not combined.empty:

        agg = combined.groupby(['arena','controller','graph_source']).agg(

            success_rate=('success','mean'),

            mean_spl=('spl','mean'),

            mean_excess=('excess_length','mean'),

            mean_path_length_cm=('path_length_cm','mean'),

            mean_blocked_steps=('blocked_steps','mean'),

            mean_target_switches=('target_switches','mean'),

            mean_forced_replans=('forced_replans','mean'),

            mean_fraction_graph_mode=('fraction_graph_mode','mean'),

            mean_graph_nodes=('graph_nodes','mean'),

            n=('success','size')

        ).reset_index()

        agg.to_csv(out / 'summary_by_arena_controller.csv', index=False)

        for metric in ['success_rate','mean_spl','mean_blocked_steps','mean_forced_replans','mean_fraction_graph_mode']:

            fig, ax = plt.subplots(figsize=(12,5))

            agg_plot = agg.copy()

            agg_plot['method'] = agg_plot['controller'].str.replace('_topograph_', '_tg_', regex=False)

            pivot = agg_plot.pivot_table(index='method', columns='arena', values=metric, aggfunc='mean')

            pivot.plot(kind='bar', ax=ax)

            ax.set_title(metric); ax.set_ylabel(metric)

            if metric in ['success_rate','mean_spl','mean_fraction_graph_mode']:

                ax.set_ylim(0, 1.05)

            fig.tight_layout(); fig.savefig(out / f'bar_{metric}.png', dpi=180); plt.close(fig)

    zip_path = out.with_suffix('.zip')

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:

        for fp in out.rglob('*'):

            zf.write(fp, fp.relative_to(out.parent))

    return str(zip_path)

print(' v2 loaded: topology-preserving PC/SC/place graphs + reverse wavefront arbitration')



In [ ]:

GRAPH_VARIANTS_V3 = [

    {'label': 'position', 'source': 'position', 'threshold': None, 'merge_radius_cm': 8.0},

    {'label': 'pc_loose', 'source': 'pc', 'threshold': 0.20, 'merge_radius_cm': 8.0},

    {'label': 'pc_medium', 'source': 'pc', 'threshold': 0.35, 'merge_radius_cm': 8.0},

    {'label': 'pc_strict', 'source': 'pc', 'threshold': 0.55, 'merge_radius_cm': 8.0},

    {'label': 'sc_loose', 'source': 'sc', 'threshold': 0.20, 'merge_radius_cm': 8.0},

    {'label': 'sc_medium', 'source': 'sc', 'threshold': 0.35, 'merge_radius_cm': 8.0},

    {'label': 'sc_strict', 'source': 'sc', 'threshold': 0.55, 'merge_radius_cm': 8.0},

    {'label': 'placebasis_loose', 'source': 'placebasis', 'threshold': 0.45, 'merge_radius_cm': 8.0},

    {'label': 'placebasis_medium', 'source': 'placebasis', 'threshold': 0.60, 'merge_radius_cm': 8.0},

    {'label': 'placebasis_strict', 'source': 'placebasis', 'threshold': 0.75, 'merge_radius_cm': 8.0},

]

BASE_CONTROLLERS_V3 = ['baseline_neural', 'wall_aware_neural']

GRAPH_CONTROLLERS_POSITION_V3 = ['topograph_geometric', 'topograph_arbitrated', 'topograph_full_hybrid']

GRAPH_CONTROLLERS_FEATURE_V3 = ['topograph_full_hybrid']

def build_graph_variant_v3(net, exploration_df, walls, variant, every=1):

    return build_topology_graph(

        net, exploration_df, walls,

        source=variant['source'],

        every=every,

        merge_radius_cm=variant.get('merge_radius_cm', 8.0),

        feature_threshold=variant.get('threshold', None),

    )

def _nearest_node_by_position(graph, pos):

    if graph.n_nodes == 0:

        return None, float('inf')

    pts = np.vstack(graph.positions)

    d = np.linalg.norm(pts - np.asarray(pos)[None, :], axis=1)

    idx = int(np.argmin(d))

    return idx, float(d[idx])

def _nearest_node_by_feature(graph, feature):

    if graph.n_nodes == 0 or len(graph.features) == 0:

        return None, np.nan

    sims = np.array([cosine_similarity(feature, f) for f in graph.features], dtype=float)

    idx = int(np.argmax(sims))

    return idx, float(sims[idx])

def graph_retrieval_diagnostics_v3(net, exploration_df, graph, source, every=10, max_samples=300):

    if source == 'position' or graph.n_nodes == 0:

        return {

            'retrieval_source': source,

            'n_samples': 0,

            'feature_position_node_agreement': np.nan,

            'mean_feature_node_spatial_error_cm': np.nan,

            'median_feature_node_spatial_error_cm': np.nan,

            'mean_feature_cosine': np.nan,

        }

    enc = PlaceBasisEncoder(walls=graph.walls, P=graph.P, spacing_cm=15.0, sigma_cm=15.0) if source == 'placebasis' else None

    rows = exploration_df.iloc[::max(1, every)]

    if len(rows) > max_samples:

        rows = rows.sample(max_samples, random_state=0)

    agree = []

    spatial_errors = []

    sims = []

    for _, row in rows.iterrows():

        pos = np.array([row.x, row.y], dtype=float)

        heading = float(row.heading)

        feat = feature_for_graph(net, pos, heading, source, enc)

        p_idx, _ = _nearest_node_by_position(graph, pos)

        f_idx, sim = _nearest_node_by_feature(graph, feat)

        if f_idx is None or p_idx is None:

            continue

        agree.append(int(f_idx == p_idx))

        spatial_errors.append(float(np.linalg.norm(graph.positions[f_idx] - pos)))

        sims.append(sim)

    if not spatial_errors:

        return {

            'retrieval_source': source,

            'n_samples': 0,

            'feature_position_node_agreement': np.nan,

            'mean_feature_node_spatial_error_cm': np.nan,

            'median_feature_node_spatial_error_cm': np.nan,

            'mean_feature_cosine': np.nan,

        }

    return {

        'retrieval_source': source,

        'n_samples': len(spatial_errors),

        'feature_position_node_agreement': float(np.mean(agree)),

        'mean_feature_node_spatial_error_cm': float(np.mean(spatial_errors)),

        'median_feature_node_spatial_error_cm': float(np.median(spatial_errors)),

        'mean_feature_cosine': float(np.mean(sims)),

    }

def run__graph_v3(seed=0, arena='uwall', seconds_explore=60.0, max_seconds_nav=26.0,

                               cue_mode='perimeter_plus_wall', graph_every=1,

                               graph_variants=GRAPH_VARIANTS_V3):

    walls = make_wall_segments_v2(arena)

    cues = make_cues(cue_mode)

    net = BurgessOkeefeNetwork(cues, P=P, seed=seed)

    start_explore = np.array([25.0, 25.0], dtype=float)

    explore_df = net.explore_environment(seconds=seconds_explore, start_pos=start_explore, walls=walls)

    goal_pos = default_goal_for_arena_v2(arena)

    final_memory = net.learn_goal('goal', goal_pos)

    starts = starts_for_arena_v2(arena)

    graphs = {}

    graph_meta = {}

    diagnostics = []

    for variant in graph_variants:

        label = variant['label']

        graph = build_graph_variant_v3(net, explore_df, walls, variant, every=graph_every)

        graphs[label] = graph

        graph_meta[label] = dict(variant)

        diag = graph_retrieval_diagnostics_v3(net, explore_df, graph, variant['source'], every=8)

        diag.update({

            'seed': seed,

            'arena': arena,

            'graph_label': label,

            'graph_source': variant['source'],

            'feature_threshold': variant.get('threshold', np.nan),

            'merge_radius_cm': variant.get('merge_radius_cm', np.nan),

            'graph_nodes': graph.n_nodes,

            'graph_edges': sum(len(v) for v in graph.edges.values()) // 2,

        })

        diagnostics.append(diag)

    waypoint_dict = make_lwall_waypoints(P=P) if arena == 'lwall' else {}

    waypoint_memories = learn_waypoint_memories(net, waypoint_dict, walls=walls, P=P) if waypoint_dict else None

    rows = []

    navs = {}

    base_controllers = list(BASE_CONTROLLERS_V3)

    if arena == 'lwall':

        base_controllers.append('waypoint_memory_hybrid')

    for controller in base_controllers:

        navs[controller] = []

        for si, st in enumerate(starts):

            h0 = net.rng.uniform(-np.pi, np.pi)

            df = navigate_with_target_manager(net, st, final_memory, waypoint_dict, waypoint_memories,

                                              start_heading=h0, max_seconds=max_seconds_nav,

                                              walls=walls, controller=controller)

            navs[controller].append(df)

            opt = shortest_path_reference(st, goal_pos, walls, P)

            summary = summarize_nav_dataframe(df, goal_pos, st, optimal_len=opt, P=P)

            summary.update({

                'seed': seed, 'arena': arena, 'controller': controller,

                'graph_label': 'none', 'graph_source': 'none',

                'feature_threshold': np.nan,

                'start_idx': si, 'graph_nodes': 0, 'graph_edges': 0,

                'mean_goal_reliability': float(df.goal_vector_reliability.mean()) if 'goal_vector_reliability' in df else np.nan,

                'direct_neural_block_ratio': float(df.blocked_direct_neural_step.mean()) if 'blocked_direct_neural_step' in df else np.nan,

            })

            rows.append(summary)

    for label, graph in graphs.items():

        source = graph_meta[label]['source']

        controllers = GRAPH_CONTROLLERS_POSITION_V3 if label == 'position' else GRAPH_CONTROLLERS_FEATURE_V3

        for controller in controllers:

            name = f'{label}_{controller}'

            navs[name] = []

            for si, st in enumerate(starts):

                h0 = net.rng.uniform(-np.pi, np.pi)

                df = navigate_with_topology_graph(net, st, final_memory, graph,

                                                  start_heading=h0, max_seconds=max_seconds_nav,

                                                  walls=walls, controller=controller)

                navs[name].append(df)

                opt = shortest_path_reference(st, goal_pos, walls, P)

                summary = summarize_nav_dataframe(df, goal_pos, st, optimal_len=opt, P=P)

                summary.update({

                    'seed': seed, 'arena': arena, 'controller': name,

                    'graph_label': label, 'graph_source': source,

                    'feature_threshold': graph_meta[label].get('threshold', np.nan),

                    'start_idx': si, 'graph_nodes': graph.n_nodes,

                    'graph_edges': sum(len(v) for v in graph.edges.values()) // 2,

                    'mean_goal_reliability': float(df.goal_vector_reliability.mean()) if 'goal_vector_reliability' in df else np.nan,

                    'direct_neural_block_ratio': float(df.blocked_direct_neural_step.mean()) if 'blocked_direct_neural_step' in df else np.nan,

                })

                rows.append(summary)

    return {

        'net': net,

        'graphs': graphs,

        'graph_meta': graph_meta,

        'graph_diagnostics': pd.DataFrame(diagnostics),

        'explore_df': explore_df,

        'goal': final_memory,

        'navs': navs,

        'summary': pd.DataFrame(rows),

        'walls': walls,

        'cues': cues,

        'arena': arena,

        'seed': seed,

    }

def save__v3_outputs(output_dir, results_by_key):

    out = Path(output_dir); out.mkdir(parents=True, exist_ok=True)

    all_summaries = []

    all_graph_diagnostics = []

    for key, res in results_by_key.items():

        arena, seed = key

        sub = out / f'{arena}_seed{seed}'

        sub.mkdir(parents=True, exist_ok=True)

        res['summary'].to_csv(sub / 'summary.csv', index=False)

        res['explore_df'].to_csv(sub / 'exploration.csv', index=False)

        res['graph_diagnostics'].to_csv(sub / 'graph_retrieval_diagnostics.csv', index=False)

        all_summaries.append(res['summary'])

        all_graph_diagnostics.append(res['graph_diagnostics'])

        for label, graph in res['graphs'].items():

            graph.to_dataframe(goal_pos=res['goal'].position).to_csv(sub / f'topology_graph_nodes_{label}.csv', index=False)

            fig, ax = plt.subplots(figsize=(6,6))

            plot_topology_graph_overlay(graph, res['goal'], starts=starts_for_arena_v2(arena), walls=res['walls'], ax=ax,

                                        title=f'{arena} seed={seed}: {label}')

            fig.tight_layout(); fig.savefig(sub / f'topology_graph_overlay_{label}.png', dpi=180); plt.close(fig)

        selected = []

        for controller in res['navs'].keys():

            if controller in ['baseline_neural','wall_aware_neural','waypoint_memory_hybrid']:

                selected.append(controller)

            elif controller in ['position_topograph_geometric','position_topograph_full_hybrid']:

                selected.append(controller)

            elif controller.endswith('_topograph_full_hybrid') and any(x in controller for x in ['pc_medium','sc_medium','placebasis_medium']):

                selected.append(controller)

        for controller in selected:

            navs = res['navs'][controller]

            fig, ax = plt.subplots(figsize=(6,6))

            ax.set_xlim(-5,P.env_size_cm+5); ax.set_ylim(-5,P.env_size_cm+5); ax.set_aspect('equal')

            ax.plot([0,P.env_size_cm,P.env_size_cm,0,0],[0,0,P.env_size_cm,P.env_size_cm,0], 'k-', lw=2)

            for w in res['walls']:

                ax.plot(w[:,0], w[:,1], 'k-', lw=5, solid_capstyle='round')

            for i, df in enumerate(navs):

                if len(df) == 0:

                    continue

                ax.plot(df.x, df.y, lw=1.2)

                ax.scatter([df.x.iloc[0]],[df.y.iloc[0]], s=18)

            g = res['goal'].position

            ax.scatter([g[0]],[g[1]], marker='x', s=120, label='goal')

            ax.set_title(f'{arena} seed={seed}: {controller}')

            ax.set_xlabel('x, cm'); ax.set_ylabel('y, cm')

            fig.tight_layout(); fig.savefig(sub / f'trajectories_{controller}.png', dpi=180); plt.close(fig)

    combined = pd.concat(all_summaries, ignore_index=True) if all_summaries else pd.DataFrame()

    diag = pd.concat(all_graph_diagnostics, ignore_index=True) if all_graph_diagnostics else pd.DataFrame()

    combined.to_csv(out / 'all_runs_summary.csv', index=False)

    diag.to_csv(out / 'all_graph_retrieval_diagnostics.csv', index=False)

    if not combined.empty:

        agg = combined.groupby(['arena','controller','graph_label','graph_source','feature_threshold']).agg(

            success_rate=('success','mean'),

            mean_spl=('spl','mean'),

            mean_excess=('excess_length','mean'),

            mean_path_length_cm=('path_length_cm','mean'),

            mean_blocked_steps=('blocked_steps','mean'),

            mean_target_switches=('target_switches','mean'),

            mean_forced_replans=('forced_replans','mean'),

            mean_fraction_graph_mode=('fraction_graph_mode','mean'),

            mean_graph_nodes=('graph_nodes','mean'),

            mean_graph_edges=('graph_edges','mean'),

            mean_goal_reliability=('mean_goal_reliability','mean'),

            mean_direct_neural_block_ratio=('direct_neural_block_ratio','mean'),

            n=('success','size')

        ).reset_index().sort_values(['arena','success_rate','mean_spl'], ascending=[True,False,False])

        agg.to_csv(out / 'summary_by_arena_controller.csv', index=False)

        for metric in ['success_rate','mean_spl','mean_fraction_graph_mode','mean_forced_replans','mean_direct_neural_block_ratio']:

            fig, ax = plt.subplots(figsize=(14,5))

            plot_df = agg.copy()

            plot_df['method'] = plot_df['controller'].str.replace('_topograph_', '_tg_', regex=False)

            pivot = plot_df.pivot_table(index='method', columns='arena', values=metric, aggfunc='mean')

            pivot.plot(kind='bar', ax=ax)

            ax.set_title(metric); ax.set_ylabel(metric)

            if metric in ['success_rate','mean_spl','mean_fraction_graph_mode','mean_direct_neural_block_ratio']:

                ax.set_ylim(0, 1.05)

            fig.tight_layout(); fig.savefig(out / f'bar_{metric}.png', dpi=180); plt.close(fig)

        feature_rows = agg[agg.graph_source.isin(['pc','sc','placebasis'])].copy()

        if not feature_rows.empty:

            fig, ax = plt.subplots(figsize=(10,5))

            for src in ['pc','sc','placebasis']:

                sub = feature_rows[(feature_rows.graph_source == src) & (feature_rows.controller.str.endswith('topograph_full_hybrid'))]

                if not sub.empty:

                    d = sub.groupby('feature_threshold').success_rate.mean().reset_index()

                    ax.plot(d.feature_threshold, d.success_rate, marker='o', label=src)

            ax.set_xlabel('feature cosine threshold'); ax.set_ylabel('success rate'); ax.set_ylim(0,1.05)

            ax.set_title('Feature threshold ablation')

            ax.legend(); fig.tight_layout(); fig.savefig(out / 'feature_threshold_ablation_success.png', dpi=180); plt.close(fig)

    zip_path = out.with_suffix('.zip')

    with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:

        for fp in out.rglob('*'):

            zf.write(fp, fp.relative_to(out.parent))

    return str(zip_path)

print(' loaded: , feature thresholds, retrieval diagnostics')



In [ ]:

SEEDS = [0, 1, 2]

ARENAS = ['open', 'lwall', 'uwall', 'scorridor']

SECONDS_EXPLORE = 60.0

MAX_SECONDS_NAV = 28.0

GRAPH_EVERY = 1

GRAPH_VARIANTS = GRAPH_VARIANTS_V3

print('Seeds:', SEEDS)

print('Arenas:', ARENAS)

print('Explore seconds:', SECONDS_EXPLORE)

print('Graph variants:', [v['label'] for v in GRAPH_VARIANTS])



In [ ]:

results_v3 = {}

for arena in ARENAS:

    for seed in SEEDS:

        print(f'\n=== Running arena={arena}, seed={seed} ===')

        res = run__graph_v3(

            seed=seed,

            arena=arena,

            seconds_explore=SECONDS_EXPLORE,

            max_seconds_nav=MAX_SECONDS_NAV,

            graph_every=GRAPH_EVERY,

            graph_variants=GRAPH_VARIANTS,

        )

        results_v3[(arena, seed)] = res

        short = res['summary'].groupby(['controller','graph_label','graph_source']).agg(

            success_rate=('success','mean'),

            mean_spl=('spl','mean'),

            mean_path=('path_length_cm','mean'),

            mean_graph_mode=('fraction_graph_mode','mean'),

            n=('success','size')

        ).reset_index().sort_values(['success_rate','mean_spl'], ascending=False)

        display(short.head(20))



In [ ]:

all_summary_v3 = pd.concat([r['summary'] for r in results_v3.values()], ignore_index=True)

agg_v3 = all_summary_v3.groupby(['arena','controller','graph_label','graph_source','feature_threshold']).agg(

    success_rate=('success','mean'),

    mean_spl=('spl','mean'),

    mean_excess=('excess_length','mean'),

    mean_path_length_cm=('path_length_cm','mean'),

    mean_blocked_steps=('blocked_steps','mean'),

    mean_target_switches=('target_switches','mean'),

    mean_forced_replans=('forced_replans','mean'),

    mean_fraction_graph_mode=('fraction_graph_mode','mean'),

    mean_graph_nodes=('graph_nodes','mean'),

    mean_graph_edges=('graph_edges','mean'),

    mean_goal_reliability=('mean_goal_reliability','mean'),

    mean_direct_neural_block_ratio=('direct_neural_block_ratio','mean'),

    n=('success','size')

).reset_index().sort_values(['arena','success_rate','mean_spl'], ascending=[True,False,False])

display(agg_v3)

all_diag_v3 = pd.concat([r['graph_diagnostics'] for r in results_v3.values()], ignore_index=True)

display(all_diag_v3.sort_values(['arena','graph_source','feature_threshold']).head(50))



In [ ]:

for metric in ['success_rate','mean_spl','mean_fraction_graph_mode','mean_forced_replans','mean_direct_neural_block_ratio']:

    fig, ax = plt.subplots(figsize=(14,5))

    plot_df = agg_v3.copy()

    plot_df['method'] = plot_df['controller'].str.replace('_topograph_', '_tg_', regex=False)

    pivot = plot_df.pivot_table(index='method', columns='arena', values=metric, aggfunc='mean')

    pivot.plot(kind='bar', ax=ax)

    ax.set_title(metric)

    if metric in ['success_rate','mean_spl','mean_fraction_graph_mode','mean_direct_neural_block_ratio']:

        ax.set_ylim(0,1.05)

    plt.tight_layout()

    plt.show()

fig, ax = plt.subplots(figsize=(10,5))

feature_rows = agg_v3[agg_v3.graph_source.isin(['pc','sc','placebasis'])].copy()

for src in ['pc','sc','placebasis']:

    sub = feature_rows[(feature_rows.graph_source == src) & (feature_rows.controller.str.endswith('topograph_full_hybrid'))]

    if not sub.empty:

        d = sub.groupby('feature_threshold').success_rate.mean().reset_index()

        ax.plot(d.feature_threshold, d.success_rate, marker='o', label=src)

ax.set_xlabel('feature cosine threshold')

ax.set_ylabel('success rate')

ax.set_ylim(0,1.05)

ax.set_title('Feature threshold ablation')

ax.legend()

plt.tight_layout()

plt.show()



In [ ]:

zip_path = save__v3_outputs('_graph_v3_outputs', results_v3)

print('Saved:', zip_path)

import os

print('Size MB:', os.path.getsize(zip_path) / 1024 / 1024)

